# Attention

Embedding + BiLSTM 모델과 추가로 Seq2seq을 구성하여 Attention 층을 추가

## 1. 감정 분류을 위한 Attention 순환 신경망: BiLSTM + Attention

기존 Embedding + BiLSTM 모델 아키텍처에 Bahdanau Attention(Additive)을 추가

BiLSTM 층을 통과해 나온 특성 벡터를 폴링 하지 않고 Attention을 통해 문장 전체에서 가장 중요한 토큰들의 정보가 압축된 컨텍스트 벡터(context vector)를 만듦


### 단계:

1. 모델 정의: BiLSTMWithAttention 클래스를 만듭니다.

2. 데이터 준비: AIHUB 감성 대화 데이터를 로드합니다.

3. 토크나이저: SentencePiece (BPE) 토크나이저를 학습시킵니다.

4. 데이터셋/로더: PyTorch Dataset 및 DataLoader를 구축합니다.

5. 학습 및 평가: 모델을 학습시키고, 손실 및 정확도 변화를 시각화하여 어텐션의 효과를 확인합니다.

### 모델링

**임베딩 레이어 사용**
- 토큰의 정수 인덱스로 부터 Embedding 레이어를 통 정수를 해당하는 임베딩 벡터로 매핑

**Bi-LSTM (Bidirectional LSTM) 레이어**
- 입력된 임베딩 벡터를 시간 축을 따라 처리하여, 시퀀스 데이터의 문맥 정보를 학습
- 양방향 LSTM은 입력 데이터를 과거와 미래 방향으로 모두 처리해 더 풍부한 문맥 정보를 캡처

**어텐션 적용(Bahdanau Attention)**
- 일반적인 LSTM은 마지막 타임스텝의 hidden state만을 사용하거나, 모든 hidden state를 단순 평균
- LSTM이 출력한 각 타임스텝의 hidden state를 평가하여, 시퀀스에서 중요한 단어(정보)에 더 높은 가중치를 부여
- 모델은 높은 가중치를 가진 중요한 단어에 집중하여 결과를 예측

- 어텐션 과정

$$u_t = \tanh(W_{\text{att}} \cdot h_t)$$


$$a_t = \text{softmax}(W_c \cdot u_t)$$

$$c = \sum_t a_t \cdot h_t$$


<center><img src="https://drive.google.com/uc?export=view&id=1oxXb1eh8ZMcIEgAvyMiw2B4TaPm81sl8" width="600"/></center>

    1. 선형 변환 이후 tanh 활성화를 통해 어텐션 스코어를 계산하기 위한 중간 표현(u)을 생성
    2. u를 단일 유닛으로 변환하여 각의 토큰 위치에 대한 스칼라 가중치(로짓)를 뽑아냄
    3. 로짓을 소프트맥스 활성화 하여 각 토큰 위치에 대한 확률 분포인 어텐션 가중치(a)를 생성
    5. 어텐션 가중치(a)는 한 문장 내에서 가장 중요한 토큰에는 높은 값이 할당됨
    6. 시퀀스 차원의 토큰 벡터 out에 어텐션 가중치 a를 각각 곱하고 가중합(Weighted Sum) 하여, 컨텍스트 벡터(Context Vecto)를 구함
    7. 산출된 컨텍스트 벡터에는 각 배치별로 문장 전체에서 가장 중요한 토큰들의 정보가 압축


In [ ]:
# [전체 흐름 상세 설명]
# 이 코드는 양방향 LSTM(BiLSTM)과 어텐션 메커니즘을 결합한 감정 분류 모델을 구현합니다.
#
# 핵심 동작 원리:
# 1. 입력 문장의 각 단어를 임베딩 벡터로 변환
# 2. 양방향 LSTM을 통해 문장의 순방향/역방향 문맥 정보 추출
# 3. 어텐션 메커니즘으로 중요한 단어에 더 집중하여 문맥 벡터 생성
# 4. 문맥 벡터를 기반으로 최종 감정 분류 수행
#
# 각 단계에서 텐서의 형태 변화를 상세히 추적하며 모델의 동작을 이해합니다.

import torch
import torch.nn as nn

class BiLSTMWithAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size=12, num_layers=4, num_classes=2):
        """
        모델 초기화 - 각 레이어 정의

        Args:
            vocab_size: 단어 사전 크기 (예: 10,000개의 서로 다른 단어)
            embedding_dim: 각 단어를 표현할 벡터의 차원 (예: 100차원)
            hidden_size: LSTM의 은닉 상태 크기 (기본 12)
            num_layers: 쌓을 LSTM 레이어 개수 (기본 4)
            num_classes: 분류할 클래스 수 (기본 2: 긍정/부정)
        """
        super(BiLSTMWithAttention, self).__init__()

        # --- 1. 임베딩 레이어: 단어 ID → 의미 벡터 변환 ---
        # 정수 인덱스(단어 ID)를 받아 해당 단어의 의미를 표현하는 밀집 벡터로 매핑
        # 예: "사랑"=5 → [0.1, -0.2, 0.8, ...] (embedding_dim 차원)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # --- 2. 양방향 LSTM 레이어: 문장의 시퀀스 정보 처리 ---
        # 순방향(앞→뒤)과 역방향(뒤→앞)으로 문장을 두 번 읽어 풍부한 문맥 정보 추출
        self.lstm = nn.LSTM(
            input_size=embedding_dim,      # 임베딩 벡터의 차원
            hidden_size=hidden_size,       # 각 LSTM 뉴런의 출력 크기
            num_layers=num_layers,         # LSTM 레이어를 몇 층 쌓을지
            batch_first=True,              # 입력 텐서 형태: [배치크기, 시퀀스길이, 특성차원]
            bidirectional=True             # 양방향 LSTM 활성화 (은닉 상태 크기 2배 증가)
        )

        # --- 3. 어텐션 메커니즘: 중요한 단어에 선택적 집중 ---
        # Bahdanau Attention(Additive Attention) 방식 구현
        # 3-1. 어텐션 스코어 계산용 선형 레이어
        # LSTM 출력을 어텐션 계산에 적합한 공간으로 변환 (비선형 변환 준비)
        self.attention = nn.Linear(hidden_size*2, hidden_size*2)

        # 3-2. 컨텍스트 벡터 생성용 선형 레이어
        # 변환된 정보를 단일 스칼라 점수로 압축 (각 단어의 중요도 점수)
        self.context_vector = nn.Linear(hidden_size * 2, 1, bias=False)

        # --- 4. 모델 안정화 및 일반화 레이어 ---
        # 4-1. 레이어 정규화: 학습 안정화 및 수렴 속도 향상
        self.layer_norm = nn.LayerNorm(hidden_size*2)
        # 4-2. 드롭아웃: 과적합 방지를 위한 뉴런 임의 비활성화 (20%)
        self.dropout = nn.Dropout(p=0.2)

        # --- 5. 최종 분류를 위한 완전연결층 ---
        # 5-1. 첫 번째 완전연결층: 문맥 벡터를 중간 특징으로 변환
        self.fc1 = nn.Linear(hidden_size*2, hidden_size)
        # 5-2. 두 번째 완전연결층: 중간 특징을 최종 클래스 점수로 변환
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        """
        순전파 과정 - 데이터가 모델을 통과하는 전체 흐름

        Args:
            x: 입력 텐서 [배치크기, 문장길이] (단어 ID 시퀀스)

        Returns:
            output: 최종 예측 점수 [배치크기, 클래스수]
        """
        # [텐서 형태 추적 시작]
        # 입력 x: [배치크기, 문장길이]
        # 예: [[1, 25, 6, 0, 0, ...]] - 1개 배치, 여러 단어 ID (0은 패딩)
        print(f"입력 텐서 형태: {x.shape}")

        # --- 1. 임베딩 레이어 통과 ---
        # 각 단어 ID를 해당하는 임베딩 벡터로 변환
        # 형태 변화: [배치크기, 문장길이] → [배치크기, 문장길이, 임베딩차원]
        x = self.embedding(x)
        print(f"임베딩 후 형태: {x.shape}")

        # --- 2. 양방향 LSTM 통과 ---
        # 문장을 순차적으로 처리하여 각 단어 위치의 은닉 상태 출력
        # out: 모든 단어 위치의 LSTM 출력 (양방향이므로 hidden_size * 2)
        # _: 마지막 시간상의 (은닉상태, 셀상태) - 여기서는 사용하지 않음
        out, _ = self.lstm(x)
        # 형태: [배치크기, 문장길이, hidden_size * 2]
        print(f"LSTM 출력 형태: {out.shape}")

        # --- 3. 어텐션 메커니즘 적용 ---
        # 3-1. 어텐션 스코어 계산을 위한 중간 표현 생성
        # LSTM 출력을 선형 변환 후 tanh 활성화 함수 적용
        u = torch.tanh(self.attention(out))
        # 형태: [배치크기, 문장길이, hidden_size * 2] (변화 없음)
        print(f"어텐션 변환 후 형태: {u.shape}")

        # 3-2. 어텐션 가중치(Attention Weights) 계산
        # 각 단어의 중요도를 나타내는 스칼라 점수 계산 후 소프트맥스 적용
        attention_scores = self.context_vector(u)  # [배치크기, 문장길이, 1]
        a = torch.softmax(attention_scores, dim=1) # 문장 길이 차원에 대해 정규화
        # 형태: [배치크기, 문장길이, 1] (각 단어의 가중치, 합계=1)
        print(f"어텐션 가중치 형태: {a.shape}")

        # 3-3. 컨텍스트 벡터(Context Vector) 생성
        # 어텐션 가중치를 LSTM 출력에 적용하여 가중합 계산
        # a * out: 각 단어의 LSTM 출력에 해당 단어의 중요도 가중치 곱함
        # sum(dim=1): 모든 단어의 가중치 적용된 출력을 합쳐 하나의 문맥 벡터 생성
        context = (a * out).sum(dim=1)
        # 형태: [배치크기, hidden_size * 2] (문장 전체 정보가 압축된 벡터)
        print(f"컨텍스트 벡터 형태: {context.shape}")

        # --- 4. 안정화 및 정규화 ---
        # 4-1. 레이어 정규화: 문맥 벡터의 분포 안정화
        context = self.layer_norm(context)
        # 4-2. 드롭아웃: 훈련 시 일부 뉴런 무작위 비활성화로 과적합 방지
        context = self.dropout(context)

        # --- 5. 최종 분류 ---
        # 5-1. 첫 번째 완전연결층: 문맥 벡터를 더 작은 특징 공간으로 투영
        dense = self.fc1(context)
        # 형태: [배치크기, hidden_size] (중간 특징 표현)

        # 5-2. 두 번째 완전연결층: 중간 특징을 클래스별 점수로 변환
        out = self.fc2(dense)
        # 형태: [배치크기, 클래스수] (각 클래스에 대한 예측 점수)
        print(f"최종 출력 형태: {out.shape}")

        return out


# [모델 생성 및 테스트 실행]
print("=== 모델 생성 ===")
# 모델 인스턴스 생성
# - vocab_size=100: 100개의 서로 다른 단어를 가진 사전
# - embedding_dim=32: 각 단어를 32차원 벡터로 표현
# - 나머지 매개변수는 기본값 사용 (hidden_size=12, num_layers=4, num_classes=2)
model = BiLSTMWithAttention(vocab_size=100, embedding_dim=32)
print("모델 생성 완료")

print("\n=== 테스트 입력 생성 ===")
# 테스트용 입력 데이터 생성
# - [[1, 2, 3, 4]]: 1개 배치, 4개 단어로 구성된 문장
# - 각 숫자는 단어 ID (1: 첫 번째 단어, 2: 두 번째 단어 등)
input = torch.tensor([[1,2,3,4]])
print(f"테스트 입력: {input}")
print(f"테스트 입력 형태: {input.shape}")

print("\n=== 순전파 실행 ===")
# 모델에 입력을 전달하여 순전파 수행
# forward 함수 내의 print문으로 각 단계의 텐서 형태 변화 확인
output = model(input)

print("\n=== 최종 결과 ===")
# 모델의 최종 출력: 각 클래스에 대한 예측 점수 (로짓)
# 아직 소프트맥스 적용 전이므로 점수가 높은 클래스가 모델이 예측한 클래스
print(f"최종 출력 값: {output}")
print(f"최종 출력 형태: {output.shape}")

# 추가: 소프트맥스 적용하여 확률로 변환 (선택사항)
probabilities = torch.softmax(output, dim=1)
print(f"소프트맥스 적용 후 확률: {probabilities}")
print(f"예측된 클래스: {torch.argmax(output, dim=1).item()}")

lstm: torch.Size([1, 4, 24])
u: torch.Size([1, 4, 24])
a: torch.Size([1, 4, 1])
context : torch.Size([1, 24])
모델 출력 결과: tensor([[-0.2332, -0.3703]], grad_fn=<AddmmBackward0>)


### 모델 학습

[AIHUB](https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=86)에서 제공하는 한국어 감성 대화 말뭉치 데이터세트를 부분적으로 활용하여 감정 분류 학습

In [ ]:
# [전체 흐름 상세 설명]
# 이 코드는 한국어 텍스트 처리를 위한 SentencePiece 토크나이저를 훈련시키는 전체 과정입니다.
#
# 핵심 작업:
# 1. Google Colab 환경에서 train.csv 파일 업로드
# 2. 데이터 탐색 및 전처리 (감정 레이블과 질문 텍스트 추출)
# 3. SentencePiece 훈련을 위한 코퍼스 파일 생성
# 4. BPE(Byte Pair Encoding) 알고리즘을 사용한 토크나이저 모델 훈련
# 5. 특수 토큰([PAD], [UNK], [BOS], [EOS]) 설정 및 모델 저장

# --- 1부: Google Colab 환경에서 파일 업로드 ---

# Google Colab의 files 모듈을 임포트하여 사용자 로컬에서 파일 업로드 가능하게 함
from google.colab import files

# 파일 업로드 대화상자 열기
# 사용자가 자신의 컴퓨터에서 'train.csv' 파일을 선택할 수 있음
# 업로드된 파일 정보는 'uploaded' 변수에 딕셔너리 형태로 저장됨
uploaded = files.upload()

# --- 필요한 라이브러리 임포트 ---

# 데이터 처리 및 분석을 위한 pandas 라이브러리
import pandas as pd
# 수치 계산을 위한 numpy 라이브러리
import numpy as np
# SentencePiece 토크나이저 라이브러리
import sentencepiece as spm
# 딥러닝 프레임워크 PyTorch
import torch
# PyTorch의 신경망 모듈
import torch.nn as nn

Saving train.csv to train.csv


In [ ]:
# --- 2부: 데이터 로딩 및 탐색 ---

print("=== 데이터 파일 로딩 ===")
# 업로드된 train.csv 파일을 pandas DataFrame으로 읽어오기
# DataFrame은 엑셀 시트와 같은 표 형태의 데이터 구조
df = pd.read_csv('./train.csv')

print(f"데이터프레임 크기: {df.shape}")  # (행 개수, 열 개수) 출력
print(f"데이터프레임 컬럼: {df.columns.tolist()}")  # 모든 컬럼 이름 출력

# --- 3부: 감정 분석을 위한 데이터 준비 ---

print("\n=== 감정 분석용 데이터 추출 ===")
# 원본 데이터프레임에서 'label'(감정 레이블)과 'HS01'(질문 텍스트) 컬럼만 선택
# 'HS01' 컬럼 이름을 'text'로 변경하여 직관적으로 만들기
# dropna()로 결측값이 있는 행 제거 (텍스트나 레이블이 없는 데이터 제외)
sent_df = df[['label','HS01']].rename({'HS01':'text'}, axis=1).dropna()

print(f"감정 분석 데이터 크기: {sent_df.shape}")
print(f"레이블 분포:")
print(sent_df['label'].value_counts())  # 각 감정 레이블별 데이터 개수 출력

# 처리된 데이터프레임 확인
print("\n=== 감정 분석 데이터 샘플 ===")
# 상위 5개 행 출력하여 데이터 확인
sent_df.head()

,label,text
0,E1,일은 왜 해도 해도 끝이 없을까? 화가 난다.
1,E1,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.
2,E1,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...
3,E1,직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...
4,E1,얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.
...,...,...
51623,E1,나이가 먹고 이제 돈도 못 벌어 오니까 어떻게 살아가야 할지 막막해. 능력도 없고.
51624,E3,몸이 많이 약해졌나 봐. 이제 전과 같이 일하지 못할 것 같아 너무 짜증 나.
51625,E4,이제 어떻게 해야 할지 모르겠어. 남편도 그렇고 노후 준비도 안 되어서 미래가 걱정돼.
51626,E3,몇십 년을 함께 살았던 남편과 이혼했어. 그동안의 세월에 배신감을 느끼고 너무 화가 나.


In [ ]:
# --- 2부: 훈련용 코퍼스 생성 ---

# sentencepiece, pandas, os 라이브러리를 다시 가져옵니다. (코드 상단에 이미 있지만, 셀을 분리해 실행할 경우를 대비)
import sentencepiece as spm
import pandas as pd
# 'os' 라이브러리를 가져옵니다. 운영체제와 상호작용(예: 디렉터리 생성)을 위해 필요합니다.
import os

# BPE 모델 관련 파일들을 저장할 디렉터리 경로를 'bpe_path' 변수에 저장합니다.
bpe_path = f'bpe'

# os.makedirs()를 사용해 'bpe'라는 이름의 디렉터리를 생성합니다.
# exist_ok=True: 'bpe' 디렉터리가 이미 존재하더라도 오류를 발생시키지 않고 넘어갑니다.
os.makedirs(bpe_path, exist_ok=True)

# SentencePiece 훈련에 사용할 코퍼스(말뭉치) 텍스트 파일의 전체 경로를 'corpus_file' 변수에 저장합니다.
# (결과: 'bpe/krsent_corpus.txt')
corpus_file = f'{bpe_path}/krsent_corpus.txt'

# [지금 하는 일: 훈련 데이터(코퍼스) 파일 생성]
# try...except: 파일 입출력 중 발생할 수 있는 오류(예: 'train.csv' 파일이 없음)를 처리하기 위한 구문입니다.
try:
    # 'train.csv' 파일을 다시 읽어옵니다. (앞의 sent_df와는 다른 목적으로 사용)
    df = pd.read_csv(f'./train.csv')
    # 이번에는 'HS01'(질문)과 'SS01'(답변) 열을 모두 선택합니다.
    # -> 토크나이저가 '질문'과 '답변'에 사용된 모든 단어를 학습하도록 하기 위함입니다.
    # .dropna()로 비어있는 행을 제거합니다.
    df = df[['HS01', 'SS01']].dropna()

    # 사용자에게 현재 진행 상황을 알려줍니다.
    print("코퍼스 파일 생성 중...")
    # 'corpus_file' 경로('bpe/krsent_corpus.txt')에 파일을 'w'(쓰기) 모드로 엽니다.
    # encoding='utf-8': 한글이 깨지지 않도록 UTF-8 인코딩을 사용합니다.
    # 'f'라는 이름으로 이 파일을 다룹니다.
    with open(corpus_file, 'w', encoding='utf-8') as f:
        # 'HS01' (질문) 열의 모든 텍스트를 하나씩 반복합니다.
        for text in df['HS01']:
            # 각 질문 텍스트(text)를 파일(f)에 쓰고, 줄바꿈(\n)을 추가합니다.
            f.write(f"{text}\n")
        # 앞에거를 다 하면 'SS01' (답변) 열의 모든 텍스트를 하나씩 반복합니다.
        for text in df['SS01']:
            # 각 답변 텍스트(text)를 파일(f)에 쓰고, 줄바꿈(\n)을 추가합니다.
            f.write(f"{text}\n")

    # 코퍼스 파일 생성이 완료되었음을 사용자에게 알립니다.
    print(f"코퍼스 파일 저장 완료: {corpus_file}")

    # --- 3부: SentencePiece 모델 훈련 ---

    # (주석) 이 코드의 다른 부분(SPDataSet, CrossEntropyLoss)과 호환되기 위해
    # (주석) [PAD]=0, [UNK]=1, [BOS]=2, [EOS]=3 라는 '특별 토큰 ID'를 강제로 지정해야 한다는 설명입니다.

    # 훈련된 모델이 저장될 경로와 파일명의 '접두사(prefix)'를 설정합니다.
    # (결과: 'bpe/spm_krsent'. 훈련 후 'bpe/spm_krsent.model', 'bpe/spm_krsent.vocab' 파일이 생성됩니다.)
    model_prefix = f'{bpe_path}/spm_krsent'
    # 토크나이저가 만들 '단어 집합(어휘)'의 최대 크기를 8000개로 설정합니다.
    vocab_size = 8000

    # [지금 하는 일: SentencePiece 훈련 옵션 설정]
    # spm.SentencePieceTrainer.train() 함수에 전달할 설정값들을 하나의 긴 문자열로 만듭니다.
    spm_command = (
        # --input: 훈련시킬 코퍼스 파일 경로를 지정합니다.
        f'--input={corpus_file} '
        # --model_prefix: 저장될 모델 파일의 접두사를 지정합니다.
        f'--model_prefix={model_prefix} '
        # --vocab_size: 만들 어휘의 크기를 지정합니다. (위에서 8000으로 설정)
        f'--vocab_size={vocab_size} '
        # --model_type=bpe: 토큰화 알고리즘으로 BPE(Byte Pair Encoding)를 사용하도록 지정합니다.
        f'--model_type=bpe '
        # --pad_id=0: [PAD] (패딩) 토큰의 ID를 0번으로 강제 할당합니다.
        f'--pad_id=0 '
        # --unk_id=1: [UNK] (알 수 없는 단어) 토큰의 ID를 1번으로 강제 할당합니다.
        f'--unk_id=1 '
        # --bos_id=2: [BOS] (문장 시작) 토큰의 ID를 2번으로 강제 할당합니다.
        f'--bos_id=2 '
        # --eos_id=3: [EOS] (문장 끝) 토큰의 ID를 3번으로 강제 할당합니다.
        f'--eos_id=3 '
        # --pad_piece=[PAD]: 0번 ID에 해당하는 토큰의 '문자열'을 '[PAD]'로 설정합니다.
        f'--pad_piece=[PAD] '
        # --unk_piece=[UNK]: 1번 ID에 해당하는 토큰의 문자열을 '[UNK]'로 설정합니다.
        f'--unk_piece=[UNK] '
        # --bos_piece=[BOS]: 2번 ID에 해당하는 토큰의 문자열을 '[BOS]'로 설정합니다.
        f'--bos_piece=[BOS] '
        # --eos_piece=[EOS]: 3번 ID에 해당하는 토큰의 문자열을 '[EOS]'로 설정합니다.
        f'--eos_piece=[EOS]'
    )

    # 사용자에게 훈련 시작을 알립니다.
    print("SentencePiece 모델 훈련 시작...")
    # [지금 하는 일: 훈련 실행]
    # SentencePiece 훈련기(Trainer)의 train 메소드를 호출합니다.
    # 위에서 만든 설정값 문자열(spm_command)을 인자로 전달하여 훈련을 실행합니다.
    # 이 과정은 'bpe/krsent_corpus.txt' 파일을 읽어 BPE 알고리즘을 수행하고,
    # 그 결과를 'bpe/spm_krsent.model'과 'bpe/spm_krsent.vocab' 파일로 저장합니다.
    spm.SentencePieceTrainer.train(spm_command)

    # 훈련이 성공적으로 완료되었음을 사용자에게 알립니다.
    print(f"모델 훈련 완료! 모델이 {model_prefix}.model 에 저장되었습니다.")

# 'try' 블록 내부에서 'train.csv' 파일을 읽을 때(pd.read_csv) 해당 파일이 존재하지 않으면,
# FileNotFoundError가 발생하며, 이 'except' 블록이 실행됩니다.
except FileNotFoundError:
  # 사용자에게 'train.csv' 파일을 찾을 수 없다는 오류 메시지를 출력합니다.
  print(f"오류: train.csv 파일을 찾을 수 없습니다.")

코퍼스 파일 생성 중...
코퍼스 파일 저장 완료: bpe/krsent_corpus.txt
SentencePiece 모델 훈련 시작...
모델 훈련 완료! 모델이 bpe/spm_krsent.model 에 저장되었습니다.


### SentencePiece 토크나이저를 이용해서 만든 PyTorch Dataset 클래스

In [ ]:
# [전체 흐름]
# 이 코드는 파이토치(PyTorch)의 'Dataset' 클래스를 상속받아, 우리만의 맞춤 데이터셋 클래스인 'SPDataSet'을 정의합니다.
# 딥러닝 모델은 '안녕하세요' 같은 텍스트나 'E1' 같은 문자열 라벨을 직접 이해할 수 없습니다.
# 이 클래스의 핵심 역할은, 1) 'train.csv'에서 읽어온 데이터(DataFrame)와 2) 앞서 훈련시킨 SentencePiece 토크나이저(sp)를 이용해,
# 텍스트 문장과 문자열 라벨을 모델이 학습할 수 있는 '숫자 텐서(Tensor)' 형태로 변환해주는 '변환기' 역할을 하는 것입니다.

# [지금 하는 일: 데이터셋 클래스 설계]
# 'torch.utils.data' 모듈에서 'Dataset' 클래스를 가져옵니다.
# 'Dataset'은 파이토치에서 데이터셋을 정의하기 위한 '설계도(추상 클래스)'이며,
# 우리는 __len__ (데이터 총 개수)와 __getitem__ (특정 인덱스(i)의 데이터) 두 함수를 반드시 구현해야 합니다.
from torch.utils.data import Dataset
# pandas 라이브러리를 'pd'라는 별칭으로 가져옵니다. (DataFrame 처리에 필요)
import pandas as pd
# numpy 라이브러리를 'np'라는 별칭으로 가져옵니다. (패딩 처리에 사용)
import numpy as np
# sentencepiece 라이브러리를 'spm'이라는 별칭으로 가져옵니다. (토크나이저 사용에 필요)
import sentencepiece as spm
# torch 라이브러리를 가져옵니다. (텐서 변환에 필요)
import torch
# torch의 신경망 모듈(nn)을 'nn'이라는 별칭으로 가져옵니다. (여기서는 직접 사용되진 않음)
import torch.nn as nn


# 'SPDataSet'이라는 이름의 새 클래스를 정의합니다. PyTorch의 'Dataset'을 상속받습니다.
class SPDataSet(Dataset):
    # 이 클래스에 대한 설명 문자열(docstring)입니다.
    """
    SentencePiece 토크나이저를 이용해서 만든 PyTorch Dataset 클래스
    모델이 학습할 수 있는 형태로 **문장(text)과 라벨(label)**을 숫자 텐서로 변환하는 역할
    """
    # [지금 하는 일: 초기화 함수 정의]
    # 이 클래스의 '생성자(초기화)' 함수입니다. 'SPDataSet(...)' 코드가 실행될 때 호출됩니다.
    # df: 학습할 데이터가 담긴 pandas DataFrame (예: train_df)
    # sp: 훈련된 SentencePiece 토크나이저 객체
    # max_len: 문장의 최대 길이를 몇으로 고정할지 (패딩/절단 기준)
    def __init__(self, df, sp, max_len):
        # 입력받은 max_len 값을 'self.max_len'이라는 클래스 변수에 저장합니다.
        # (이유: 클래스 내의 다른 함수(__getitem__, zero_pad)에서도 이 값을 사용해야 하기 때문)
        self.max_len = max_len
        # 입력받은 DataFrame을 'self.df' 변수에 저장합니다.
        self.df = df
        # 입력받은 SentencePiece 객체를 'self.sp' 변수에 저장합니다.
        self.sp = sp
        # [지금 하는 일: 라벨 인코딩용 사전 정의]
        # 'E1', 'E6' 같은 문자열 라벨을 0, 1, 2... 같은 숫자로 변환(인코딩)하기 위한 '사전(dictionary)'을 정의합니다.
        # (이유: 모델은 문자열을 이해하지 못하며, 숫자로 된 타겟값이 필요하기 때문입니다.)
        self.class_name = {'E1':0, 'E6':1, 'E3':2, 'E5':3, 'E2':4, 'E4':5}

    # [지금 하는 일: 패딩 함수 정의]
    # 'tok' (토큰화된 숫자 리스트)를 'self.max_len' 길이에 맞게 '패딩(padding)'하거나 '절단(truncating)'하는 함수입니다.
    # (이유: 모델(특히 LSTM)에 데이터를 배치(batch) 단위로 넣으려면, 모든 문장의 길이를 동일하게 맞춰야 합니다.)
    # tok: [BOS_ID, 5, 32, 12, EOS_ID] 처럼 토큰 ID가 담긴 리스트
    def zero_pad(self, tok):
        # 만약 토큰 리스트(tok)의 길이가 우리가 정한 최대 길이(max_len)보다 길거나 같다면,
        if len(tok) >= self.max_len:
            # 최대 길이(max_len)까지만 잘라내어 반환합니다. (예: max_len=10이면 0~9번 인덱스까지만)
            return tok[:self.max_len]
        # 만약 토큰 리스트의 길이가 최대 길이보다 짧다면,
        else:
            # 먼저, 'np.zeros'를 사용해 'self.max_len' 길이만큼 0으로 채워진 numpy 배열을 만듭니다. (이 0이 [PAD] 토큰 ID가 됩니다)
            padding = np.zeros(self.max_len)
            # 이 0 배열의 앞부분(0번 인덱스부터 tok의 길이만큼)을 실제 토큰 리스트(tok) 내용으로 덮어씁니다.
            # (예: max_len=10, tok=[2,5,3] -> padding은 [2., 5., 3., 0., 0., 0., 0., 0., 0., 0.] 가 됨)
            padding[:len(tok)] = tok
            # 0으로 채워진(패딩된) 배열을 반환합니다.
            return padding

    # [지금 하는 일: 데이터 반환 함수 정의]
    # Dataset 클래스의 핵심 함수. 데이터 로더가 'i'번째 데이터를 요청할 때 호출됩니다.
    # i: 0, 1, 2, ... 처럼 데이터의 인덱스(순번)
    def __getitem__(self, i):
        # 'self.df' (DataFrame)에서 'i'번째 행('iloc[i]')의 'text' 열에 있는 값을 문자열(str)로 가져옵니다.
        inp = str(self.df.iloc[i]['text'])
        # 'self.df'에서 'i'번째 행의 'label' 열에 있는 값을 가져옵니다. (예: 'E1')
        tar = self.df.iloc[i]['label']

        # --- 라벨 인코딩 ---
        # 'self.class_name' 사전을 이용해 문자열 라벨(tar)을 숫자로 변환합니다. (예: 'E1' -> 0)
        tar = self.class_name[tar]

        # --- 문장 인코딩 (토큰화) ---
        # 1. self.sp.bos_id(): 문장 시작을 알리는 '[BOS]' 토큰의 ID(숫자, 예: 2)를 가져옵니다.
        # 2. self.sp.encode_as_ids(inp): 'inp' (문자열)을 SentencePiece 토크나이저로 쪼개어 ID 리스트로 만듭니다. (예: [5, 32, 12])
        # 3. self.sp.eos_id(): 문장 끝을 알리는 '[EOS]' 토큰의 ID(숫자, 예: 3)를 가져옵니다.
        # 이 세 가지를 리스트로 합칩니다. (결과: [2, 5, 32, 12, 3])
        inp = [self.sp.bos_id()] + self.sp.encode_as_ids(inp) + [self.sp.eos_id()]

        # --- 패딩 ---
        # 위에서 만든 숫자 리스트(inp)를 'self.zero_pad' 함수에 넣어 'max_len' 길이에 맞게 0으로 채우거나 잘라냅니다.
        inp = self.zero_pad(inp)

        # 최종적으로 두 가지를 반환합니다:
        # 1. torch.Tensor(inp): 패딩까지 완료된 numpy 배열(inp)을 파이토치 '텐서'로 변환합니다. (모델의 입력값)
        # 2. tar: 숫자로 변환된 라벨. (모델의 정답값)
        return torch.Tensor(inp), tar

    # [지금 하는 일: 데이터 총 개수 반환 함수 정의]
    # Dataset 클래스의 핵심 함수. 데이터셋에 총 몇 개의 데이터가 있는지 반환합니다.
    def __len__(self):
        # 'self.df' (DataFrame)의 전체 행(row)의 개수(len)를 반환합니다.
        return (len(self.df))

### 모델이 학습하기 전에 학습 파이프라인 전체를 준비하는 과정

In [ ]:
# [전체 흐름]
# 이 코드는 이전에 정의한 'BiLSTMWithAttention' 모델과 'SPDataSet' 클래스를 이용해,
# 딥러닝 모델 '훈련(training)을 시작하기 직전의 모든 준비 과정'을 수행합니다.
# 1. 훈련에 필요한 라이브러리들을 가져옵니다.
# 2. 모델의 구조와 학습 방식을 결정하는 '하이퍼파라미터'를 설정합니다.
# 3. 이전에 훈련시킨 SentencePiece 토크나이저를 불러옵니다.
# 4. 설정값(하이퍼파라미터)을 바탕으로 'BiLSTMWithAttention' 모델의 실체를 생성(초기화)합니다.
# 5. 'SPDataSet'을 이용해 전체 데이터를 준비하고, 이를 '훈련용'과 '검증용'으로 분할합니다.
# 6. 분할된 데이터셋을 'DataLoader'로 감싸, 모델에 배치(batch) 단위로 효율적으로 공급할 수 있게 만듭니다.
# 7. 훈련을 가속화하기 위해 GPU(cuda) 사용을 설정하고 모델을 GPU로 이동시킵니다.

# [지금 하는 일: 필요 라이브러리 임포트]
# 파이토치의 'optim' 모듈을 'optim'이라는 별칭으로 가져옵니다.
# (이유: 'Adam', 'SGD' 등 모델의 가중치를 업데이트(학습)하는 '옵티마이저'를 사용하기 위해 필요합니다.)
import torch.optim as optim
# 파이토치의 'torch.utils.data' 모듈에서 'random_split'과 'DataLoader' 클래스를 가져옵니다.
# (이유: 'random_split'은 전체 데이터셋을 훈련/검증용으로 나누기 위해, 'DataLoader'는 데이터를 배치 단위로 묶어주기 위해 필요합니다.)
from torch.utils.data import random_split, DataLoader
# 'sentencepiece' 라이브러리를 'spm'이라는 별칭으로 가져옵니다.
# (이유: 이전에 훈련시킨 BPE 토크나이저 모델 파일(.model)을 불러와 사용하기 위해 필요합니다.)
import sentencepiece as spm
# 파이토치(torch) 라이브러리를 가져옵니다. (텐서 연산, GPU 설정 등에 필요)
import torch


# [지금 하는 일: 하이퍼파라미터 설정]
# (주석) '하이퍼파라미터'란, 모델이 스스로 학습하는 값(가중치)이 아니라,
# (주석) '사람'이 직접 설정해줘야 하는 값들입니다. 이 값들에 따라 모델의 성능이 크게 달라집니다.

# embedding_dim: 단어를 표현하는 벡터의 차원(크기)을 128로 설정합니다.
# (이유: 이 값이 크면 단어의 풍부한 의미를 담을 수 있지만, 모델이 무거워집니다.)
embedding_dim = 128
# max_len: 문장의 최대 길이를 60으로 설정합니다. (SPDataSet으로 전달될 값)
# (이유: 60개 토큰보다 길면 잘라내고, 짧으면 0(PAD)으로 채워 모든 문장 길이를 통일하기 위함입니다.)
max_len = 60
# hidden_size: LSTM 레이어 내부의 은닉 상태(hidden state) 벡터 크기를 256으로 설정합니다.
# (이유: 이 값이 크면 모델의 표현력이 좋아지지만(문맥을 더 잘 기억함), 계산량이 많아집니다.)
hidden_size = 256
# num_layers: LSTM 레이어를 2개 층으로 쌓아 올리도록 설정합니다.
# (이유: 1개 층보다 2개 층이 더 복잡한 패턴을 학습할 수 있습니다.)
num_layers = 2
# num_classes: 모델이 최종적으로 분류해야 할 클래스(정답)의 개수를 6으로 설정합니다.
# (이유: SPDataSet에서 'E1'~'E6'까지 6개의 라벨을 사용하기 때문입니다.)
num_classes = 6
# learning_rate (학습률): 모델이 정답을 맞추기 위해 가중치를 한 번에 얼마나 '조금씩' 수정할지 그 정도를 0.0001로 설정합니다.
# (이유: 값이 너무 크면 학습이 불안정(발산)하고, 너무 작으면 학습 속도가 매우 느려집니다.)
learning_rate = 0.0001
# num_epochs (에포크 수): 전체 훈련 데이터를 총 20번 반복해서 학습하도록 설정합니다.
num_epochs = 20
# batch_size: 훈련 데이터를 한 번에 64개씩 묶어서 모델에 넣고 학습시키도록 설정합니다.
# (이유: 1개씩 넣는 것보다 64개씩 묶어서 GPU로 병렬 처리하는 것이 훨씬 빠릅니다.)
batch_size = 64

# [지금 하는 일: 토크나이저 불러오기]
# SentencePieceProcessor 객체를 생성(초기화)합니다.
# (이유: 이 객체를 통해 '.model' 파일을 로드하여, 문장을 토큰 ID로 바꾸는 'encode' 기능을 사용할 수 있습니다.)
# model_file=...: 이전에 'bpe' 폴더에 훈련시켜 저장해 둔 'spm_krsent.model' 파일의 경로를 지정합니다.
sp = spm.SentencePieceProcessor(model_file=f'./bpe/spm_krsent.model')
# 토크나이저(sp)가 가지고 있는 전체 어휘(단어)의 개수를 'get_piece_size()' 함수로 가져와 'vocab_size' 변수에 저장합니다.
# (이유: 이 'vocab_size'는 모델의 Embedding 레이어 크기를 결정하는 데 필수적입니다. 예: 8000)
vocab_size = sp.get_piece_size()

# [지금 하는 일: 모델 초기화]
# 'BiLSTMWithAttention' 클래스(설계도)를 바탕으로 실제 모델 객체(model)를 생성합니다.
# (이유: __init__ 함수에 위에서 설정한 하이퍼파라미터들을 전달하여, 우리가 원하는 구조의 모델을 만듭니다.)
# (BiLSTMWithAttention 클래스는 첫 번째로 주신 코드에 정의되어 있어야 합니다.)
model = BiLSTMWithAttention(vocab_size, embedding_dim, hidden_size, num_layers, num_classes)

# [지금 하는 일: 데이터세트 준비 및 분할]
# 'SPDataSet' 클래스(설계도)를 바탕으로 실제 데이터셋 객체(dataset)를 생성합니다.
# (이유: DataFrame(sent_df), 토크나이저(sp), 최대 길이(max_len)를 전달하여,
# __getitem__으로 데이터를 요청할 때마다 텍스트를 숫자 텐서로 변환해주는 '변환기'를 만듭니다.)
# (sent_df는 이전 코드 셀에서 `df[['label','HS01']].rename(...)`로 만든 DataFrame이어야 합니다.)
dataset = SPDataSet(sent_df, sp, max_len)

# 데이터 분할 시 재현성을 위한 '시드(seed)'를 고정하는 'Generator' 객체를 생성합니다.
# (이유: 'manual_seed(42)'를 설정하면, 코드를 다시 실행해도 'random_split'이 항상 '똑같은 방식'으로 데이터를 섞습니다.
# 이렇게 해야 어제와 오늘의 훈련/검증셋이 동일하게 유지되어 실험 결과를 신뢰할 수 있습니다.)
generator1 = torch.Generator().manual_seed(42)

# 'random_split' 함수를 사용해 전체 'dataset'을 [0.2, 0.8] 비율로 분할합니다.
# (이유: 전체 데이터를 20%는 '검증용(test_dataset)', 80%는 '훈련용(train_dataset)'으로 나누기 위함입니다.)
# generator=generator1: 위에서 만든 시드 고정용 generator를 사용합니다.
test_dataset, train_dataset = random_split(dataset, [0.2, 0.8], generator=generator1)

# [지금 하는 일: 데이터로더(DataLoader) 생성]
# 'train_dataset'을 'DataLoader'로 감싸 'train_loader'를 생성합니다.
# (이유: 'train_loader'는 훈련 시 데이터를 'batch_size'(64)만큼 자동으로 묶어주고,
# shuffle=True: 데이터를 모델에 넣기 전에 매 에포크마다 순서를 무작위로 섞어줍니다. (과적합 방지에 중요))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# 'test_dataset'(검증용)을 'DataLoader'로 감싸 'val_loader'를 생성합니다.
# (이유: 검증 시에도 'batch_size'(64)만큼 묶어서 처리하면 효율적입니다.)
# shuffle=False: '검증' 시에는 데이터를 섞을 필요가 없으므로(순서가 상관없음) False로 설정하여 속도를 높입니다.
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# [지금 하는 일: GPU(Device) 설정]
# 훈련을 실행할 장치(device)를 설정합니다.
# torch.cuda.is_available(): 현재 환경에서 NVIDIA GPU(CUDA)를 사용할 수 있는지 확인합니다.
# (결과: GPU가 있으면 'cuda' 문자열을, 없으면 'cpu' 문자열을 'device' 변수에 저장합니다.)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# [지금 하는 일: 모델을 GPU로 이동]
# 위에서 생성한 모델(model)을 '.to(device)'를 사용해 설정된 장치(GPU 또는 CPU)로 보냅니다.
# (이유: 모델의 모든 가중치(파라미터)와 연산이 GPU에서 수행되도록 하여, 훈련 속도를 수십 배 이상 가속화하기 위함입니다.)
model = model.to(device)

### 모델 학습(train) + 평가(validation) 단계

In [ ]:
# [전체 흐름]
# 이 코드는 딥러닝 모델을 '실제로 훈련시키는' 핵심 부분입니다.
# 1. 'criterion'(손실 함수)과 'optimizer'(최적화기)를 정의합니다. (모델이 무엇을 틀렸는지 계산하고, 어떻게 고칠지 정하는 도구)
# 2. 'history' 사전을 만들어, 에포크(epoch)마다의 훈련/검증 결과를 기록할 준비를 합니다. (성적표 기록 시작)
# 3. 'num_epochs' (예: 20번) 만큼 큰 반복문(for epoch...)을 돕니다.
# 4. 각 에포크(한 바퀴)마다 두 가지 작업을 순서대로 수행합니다:
#    - (1) 훈련(Training): 'train_loader'(훈련용 데이터)를 모델에 주입하고, 정답과 비교(loss)한 뒤, 틀린 만큼 모델의 가중치를 수정(업데이트)합니다. (model.train() 모드 - 공부 모드)
#    - (2) 검증(Validation): 'val_loader'(검증용 데이터)를 모델에 주입하여, '훈련에 사용하지 않은' 데이터에 대해 모델이 얼마나 잘하는지 '평가'만 합니다. (가중치 수정 X) (model.eval() 모드 - 시험 모드)
# 5. 매 에포크의 훈련 손실, 검증 손실, 검증 정확도를 'history'에 저장하고 화면에 출력합니다.

# [지금 하는 일: 손실 함수(Criterion) 정의]
# 'nn.CrossEntropyLoss'를 'criterion'이라는 이름의 객체로 생성합니다.
# (이유: 이 함수는 다중 클래스 분류(예: 6개 감정 중 하나 맞추기) 문제에 표준적으로 사용됩니다.)
# (작동 방식: 모델이 출력한 6개의 점수(logit, 예: [0.1, -0.2, 1.5, ...])와 실제 정답(예: 2번)을 입력받아,
# 둘 사이의 '차이'(손실 값, 예: 0.85)를 계산해줍니다. 이 값이 작을수록 모델이 잘 맞춘 것입니다.)
criterion = nn.CrossEntropyLoss()

# [지금 하는 일: 옵티마이저(Optimizer) 정의]
# 'torch.optim.Adam' 옵티마이저를 'optimizer'라는 이름의 객체로 생성합니다.
# (이유: 'Adam'은 "어떻게 하면 손실(loss)을 가장 효율적으로 줄일 수 있을까?"를 계산해주는 '최적화 도구'입니다.
# 성능이 좋고 안정적이어서 널리 사용됩니다.)
# (인자 1: model.parameters() -> 이 옵티마이저가 '업데이트(수정)할 대상'이 'model'의 모든 학습 가능한 가중치임을 알려줍니다.)
# (인자 2: lr=learning_rate -> 이전에 설정한 'learning_rate'(0.0001)를 학습률로 사용하라고 알려줍니다. 즉, "가중치를 0.0001 만큼씩 조심스럽게 수정해라"는 의미입니다.)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# [지금 하는 일: 학습 기록용 사전(Dictionary) 초기화]
# 훈련 과정(train_loss)과 검증 과정(val_loss, val_acc)의 지표(metric)를
# 매 에포크마다 저장하기 위한 빈 리스트들을 가진 'history' 사전을 만듭니다.
# (이유: 훈련이 끝난 후, 이 'history' 값을 사용해 '에포크별 손실 변화' 같은 학습 그래프를 그릴 수 있습니다.)
history = {'train_loss': [],
           'val_loss': [],
           'val_acc': []
          }

# [지금 하는 일: 전체 훈련 루프(Epoch Loop) 시작]
# 'num_epochs' (예: 20)번 만큼 반복하는 외부 루프를 시작합니다.
# 'epoch' 변수에는 0, 1, 2, ..., 19 가 한 바퀴(epoch) 돌 때마다 차례대로 들어갑니다.
# (이유: '에포크(epoch)'는 전체 훈련 데이터셋을 1번 학습한 것을 의미합니다. 총 20번 반복 학습을 시도합니다.)
for epoch in range(num_epochs):
    # [지금 하는 일: 1. 훈련(Training) 모드 설정]
    # 'model.train()'을 호출하여 모델을 '훈련 모드'로 전환합니다.
    # (이유: 이 모드에서는 '드롭아웃(Dropout)'이나 '배치 정규화(BatchNorm)' 등이 활성화(ON)됩니다.
    # 이는 모델이 훈련 데이터에 과적합(overfitting)되는 것을 방지해주는 기능들입니다.)
    model.train()
    # 이번 에포크의 총 훈련 손실을 누적하기 위한 변수를 0.0으로 초기화합니다.
    # (이유: 배치마다의 손실을 계속 더해서, 나중에 평균을 내기 위함입니다.)
    total_loss = 0.0

    # [지금 하는 일: 훈련 데이터 배치(Batch) 루프 시작]
    # 'train_loader'에서 'batch_size'(예: 64)만큼 묶인 데이터 배치를 하나씩 꺼내옵니다.
    # (예: 12800개 데이터면 64개씩 200번 반복)
    # 'input'에는 64개의 문장 텐서가, 'labels'에는 64개의 정답 라벨이 들어갑니다.
    for input, labels in train_loader:
        # --- 1-1. 데이터 준비 (GPU로 이동) ---
        # 'input' 텐서(SPDataSet이 반환한 텐서)를 '.long()' (64비트 정수) 타입으로 변환하고,
        # '.to(device)' (예: 'cuda')를 사용해 GPU 메모리로 보냅니다.
        # (이유: 텐서의 기본 타입(float)이 아닌 'long' 타입이 'nn.Embedding' 레이어의 입력으로 필요합니다.)
        # (모양: [batch_size, seq_len] -> [64, 60])
        input_ids = input.long().to(device)
        # 'labels' 텐서도 '.to(device)' (예: 'cuda')로 보냅니다.
        # (이유: 모델의 출력(GPU에 있음)과 손실을 계산하려면, 라벨도 같은 장치(GPU)에 있어야 합니다.)
        labels = labels.to(device)

        # --- 1-2. 순전파(Forward Pass) ---
        # 준비된 'input_ids'를 모델(model)에 입력하여 'forward' 함수를 실행하고,
        # 그 결과(예측값)를 'outputs' 변수에 받습니다.
        # (모양: [64, 60] 입력 -> [64, num_classes(6)] 출력)
        outputs = model(input_ids)
        # 'criterion'(CrossEntropyLoss)을 사용해 모델의 예측값(outputs)과 실제 정답(labels) 사이의 '손실(loss)'을 계산합니다.
        # (이것이 "모델이 이번 배치에서 얼마나 틀렸는가"를 나타내는 값입니다.)
        loss = criterion(outputs, labels)

        # --- 1-3. 역전파(Backpropagation) 및 가중치 업데이트 (핵심 학습 단계) ---
        # 'optimizer.zero_grad()'를 호출하여, 이전에 계산되었던 모든 가중치의 '기울기(gradient)'를 0으로 초기화(리셋)합니다.
        # (이유: 이걸 하지 않으면, 이전 배치의 기울기가 현재 배치의 기울기에 '누적'되어 학습이 망가지게 됩니다.)
        optimizer.zero_grad()
        # 'loss.backward()'를 호출하여, 방금 계산된 'loss' 값을 바탕으로 모델의 모든 가중치에 대해 "손실을 줄이려면 이 가중치를 어느 방향으로 얼마나 바꿔야 하는지" (즉, '기울기')를 '역전파' 방식으로 계산합니다.
        loss.backward()
        # 'optimizer.step()'을 호출하여, 'Adam' 옵티마이저가 'backward()'로 계산된 기울기(gradient)와 'learning_rate'를 바탕으로 모델의 가중치를 '실제로 업데이트'(수정)합니다.
        # (이것이 '학습'이 일어나고 모델이 "똑똑해지는" 실제 순간입니다.)
        optimizer.step()

        # --- 1-4. 손실 누적 ---
        # 이번 배치에서 계산된 'loss' 값(텐서)에서 '.item()'을 사용해 파이썬 숫자(float) 값만 추출하여, 'total_loss'에 더해줍니다.
        # (이유: 텐서 자체를 더하면 GPU 메모리가 계속 쌓일 수 있으므로(그래프가 유지됨), 순수한 '값'만 빼내어 누적합니다.)
        total_loss += loss.item()

    # [지금 하는 일: 에포크 훈련 결과 계산 및 저장]
    # (훈련 배치 루프가 끝난 후, 즉 훈련 데이터 1바퀴를 다 돈 후)
    # 누적된 'total_loss'를 'train_loader'의 총 배치 개수(len(train_loader))로 나누어, 이번 에포크의 '평균 훈련 손실'을 계산합니다.
    avg_loss = total_loss / len(train_loader)
    # 계산된 평균 손실을 'history' 사전의 'train_loss' 리스트에 추가(append)합니다.
    history['train_loss'].append(avg_loss)
    # 현재 에포크 번호(epoch+1, 0부터 시작했으므로 +1)와 계산된 평균 훈련 손실(avg_loss)을 화면에 출력합니다. (:.4f는 소수점 4자리까지)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

    # [지금 하는 일: 2. 검증(Validation) 모드 설정]
    # 'model.eval()'을 호출하여 모델을 '평가 모드'로 전환합니다.
    # (이유: 이 모드에서는 '드롭아웃(Dropout)' 등이 비활성화(OFF)됩니다.
    # 훈련 때와 달리, 예측 성능을 정확하고 일관되게 측정하기 위해 필요합니다. (시험 볼 때는 찍기(드롭아웃)하면 안 됨))
    model.eval()
    # 이번 에포크의 총 '검증 손실'을 누적할 변수를 0.0으로 초기화합니다.
    total_val_loss = 0.0
    # 이번 에포크의 총 '검증 정확도'를 누적할 변수를 0.0으로 초기화합니다.
    total_val_acc = 0.0

    # [지금 하는 일: 검증 데이터 배치 루프 시작]
    # 'torch.no_grad()' 블록 안에서 코드를 실행합니다.
    # (이유: '평가' 시에는 'loss.backward()' (역전파)를 할 필요가 없으므로, 기울기 계산을 '비활성화'합니다.
    # 이렇게 하면 메모리 사용량을 획기적으로 줄이고 계산 속도를 높일 수 있습니다.)
    with torch.no_grad():
        # 'val_loader'(검증 데이터 로더)에서 'batch_size'만큼 묶인 데이터 배치를 하나씩 꺼내옵니다.
        for input, labels in val_loader:
            # --- 2-1. 데이터 준비 (GPU로 이동) ---
            # 훈련 때와 동일하게, 검증용 'input' 텐서를 '.long()' 타입으로 변환하고 GPU('.to(device)')로 보냅니다.
            input_ids = input.long().to(device)
            # 검증용 'labels' 텐서도 GPU('.to(device)')로 보냅니다.
            labels = labels.to(device)

            # --- 2-2. 순전파(Forward Pass) ---
            # 준비된 'input_ids'를 모델(평가 모드)에 입력하여 'outputs' (예측값)을 받습니다.
            # (모양: [64, 6] 출력)
            outputs = model(input_ids)
            # 'criterion'을 사용해 예측값(outputs)과 정답(labels) 사이의 '검증 손실(loss)'을 계산합니다.
            loss = criterion(outputs, labels)
            # 계산된 검증 손실 값을 '.item()'으로 추출하여 'total_val_loss'에 누적합니다.
            total_val_loss += loss.item()

            # --- 2-3. 정확도(Accuracy) 계산 ---
            # (1) outputs.argmax(dim=1): 모델의 출력(outputs, [64, 6])에서
            #     dim=1(클래스 차원)을 기준으로 가장 큰 값의 '인덱스(index)'를 찾습니다.
            #     (예: [0.1, -0.2, 1.5, 0.3, ...] -> 2)
            #     (이것이 모델의 '최종 예측'입니다. 모양: [64])
            # (2) ... == labels: 모델의 예측(1)과 실제 정답(labels, 모양: [64])을 비교합니다.
            #     (결과: [True, False, True, ...] 처럼 맞았는지 틀렸는지(boolean)를 담은 텐서. 모양: [64])
            # (3) .sum(): (2)의 결과에서 'True' (맞춘 것)의 개수를 셉니다. (예: 50개)
            # (4) / len(labels): (3)의 개수를 현재 배치의 크기(len(labels), 예: 64)로 나누어 '이번 배치의 평균 정확도'를 계산합니다. (예: 50/64 = 0.78125)
            acc = (outputs.argmax(dim=1) == labels).sum() / len(labels)
            # 계산된 '배치 정확도(acc)'를 'total_val_acc'에 누적합니다.
            total_val_acc += acc

    # [지금 하는 일: 에포크 검증 결과 계산 및 저장]
    # (검증 배치 루프가 끝난 후, 즉 검증 데이터 1바퀴를 다 돈 후)
    # 누적된 'total_val_loss'를 'val_loader'의 총 배치 개수(len)로 나누어, 이번 에포크의 '평균 검증 손실'을 계산합니다.
    avg_val_loss = total_val_loss / len(val_loader)
    # 누적된 'total_val_acc'를 'val_loader'의 총 배치 개수(len)로 나누어, 이번 에포크의 '평균 검증 정확도'를 계산합니다.
    avg_val_acc = total_val_acc / len(val_loader)
    # 계산된 평균 검증 손실을 'history' 사전의 'val_loss' 리스트에 추가합니다.
    history['val_loss'].append(avg_val_loss)
    # 계산된 평균 검증 정확도를 '.cpu().numpy()'를 이용해 GPU 텐서 -> CPU 텐서 -> Numpy 값으로 변환한 뒤, 'history'의 'val_acc' 리스트에 추가합니다.
    # (이유: 'history'는 파이썬 리스트이므로, GPU에 있는 텐서를 바로 저장하는 것보다 numpy/파이썬 숫자로 변환해 저장하는 것이 안전합니다.)
    history['val_acc'].append(avg_val_acc.cpu().numpy())

    # [지금 하는 일: 최종 결과 출력]
    # 이번 에포크의 훈련과 검증이 모두 끝났으므로,
    # 계산된 평균 검증 손실(avg_val_loss)과 평균 검증 정확도(avg_val_acc)를 화면에 출력합니다.
    # (이유: 사용자가 훈련 과정을 모니터링하고 모델이 잘 학습되고 있는지(예: 검증 손실이 줄고, 정확도가 오르는지) 확인할 수 있습니다.)
    print(f"Validation Loss: {avg_val_loss:.4f}, Accuracy: {avg_val_acc:.4f}")

Streaming output truncated to the last 5000 lines.
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size([64, 512])
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size([64, 512])
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size([64, 512])
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size([64, 512])
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size([64, 512])
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size([64, 512])
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size([64, 512])
lstm: torch.Size([64, 60, 512])
u: torch.Size([64, 60, 512])
a: torch.Size([64, 60, 1])
context : torch.Size(

> 단순 RNN 위에 Attention을 추가했을때 성능 향상은 있지만 여전히 과적합에는 취약하다


## 2. Seq2seq + Attention (질의응답 챗봇)

간단한 Seq2seq 모델을 구축하여 질의로 부터 응답을 생성해내는 모델 아케텍처를 만듭니다.

이때 Attention을 추가하여 입력 시퀀스의 각 토큰에 대한 중요도를 계산하여 모든 토큰을 균등하게 처리하는 것이 아니라, 중요한 토큰에 더 집중할 수 있게 합니다.

### 단계:

1. 모델 정의 (Encoder): 입력 문장을 인코딩하는 Encoder 클래스를 만듭니다.

2. 모델 정의 (Decoder): 인코더의 정보와 어텐션을 활용해 응답을 생성하는 Decoder 클래스를 만듭니다.

3. 데이터셋/로더: Seq2seq용 Dataset을 만듭니다. (입력: 질문, 타겟: [BOS] + 응답 + [EOS])

4. 학습 (Teacher Forcing): '교사 강요(Teacher Forcing)' 방식을 사용하여 Seq2seq 모델을 학습시킵니다.

5. 추론 (Inference): 학습된 모델로 실제 새로운 문장을 생성(Greedy Search)해봅니다.

### 모델링

<center><img src="https://drive.google.com/uc?export=view&id=1JMeLDYYz6bruFbNfyO76ruWel1rRM5rq" width="800"/></center>


**인코더(Encoder)**
- 입력 시퀀스를 임베딩 후 양방향 LSTM으로 인코딩, 전체 은닉 상태들을 출력

**디코더(Decoder)**
- 이전 시점의 단어(또는 임베딩)와 은닉 상태로 새로운 은닉 상태를 만든 뒤, 인코더 은닉 상태와 어텐션을 통해 컨텍스트 벡터를 구해 최종 출력

**어텐션**
- Bahdanau 어텐션 형태의 단순 버전으로, 인코더 출력을 디코더 은닉 상태에 맞춰 가중합


#### 인코더 구성

인코더는 입력된 텍스트에 대한 가중치를 만들어내는 작업만 진행하기 때문에 기존 임베딩+RNN 모델 구조와 유사하게 구현됩니다.

디코더와 연결을 위해 LSTM의 출력 결과와 히든 상태를 전부 return 합니다.

In [ ]:
# 파이토치 라이브러리를 가져옵니다. 딥러닝 모델 구축에 필요합니다.
import torch
# 파이토치의 신경망 모듈(nn)을 'nn'이라는 별칭으로 가져옵니다. (nn.Module, nn.Embedding, nn.LSTM 등)
import torch.nn as nn
# 파이토치의 함수형 API를 'f'라는 별칭으로 가져옵니다. (여기서는 직접 사용되진 않음)
import torch.nn.functional as f

# [전체 흐름 1: 'Encoder' (인코더) 클래스 설계]
# 'Encoder' 클래스를 정의합니다. 'nn.Module'을 상속받아 파이토치 모델로 작동하게 합니다.
#
# (주석) '인코더'는 Seq2seq 모델의 '독자(Reader)'입니다.
# (주석) 1. 입력된 문장(질문, 예: "오늘 날씨 어때?")을 토큰 ID 리스트(예: [1, 2, 3, 4])로 받습니다.
# (주석) 2. 이 ID 리스트를 단어의 의미가 담긴 '임베딩 벡터'의 시퀀스로 변환합니다.
# (주석) 3. 'LSTM'이라는 순환 신경망(RNN)이 이 벡터 시퀀스를 순서대로 읽어 내려갑니다.
# (주석) 4. 문장을 다 읽은 후, 'Encoder'는 두 가지 핵심 정보를 출력합니다.
# (주석)    - (1) 'outputs' (모든 시점의 은닉 상태): "오늘", "날씨", "어때?" 각 단어를 읽었을 때의 '중간 생각'들. (이것이 나중에 'Attention'이 참고할 자료가 됩니다.)
# (주석)    - (2) 'h, c' (최종 은닉 상태/셀 상태): 문장 전체를 요약한 '최종 생각' (Context Vector). (이것이 'Decoder'에게 전달될 문맥입니다.)
class Encoder(nn.Module):
    # [지금 하는 일: 인코더 초기화 함수(생성자) 정의]
    # 인코더 객체가 생성될 때(예: Encoder(...)) 호출되며, 필요한 '부품(레이어)'들을 미리 정의합니다.
    # vocab_size: 단어 집합의 크기 (예: 8000개)
    # embedding_dim: 각 단어를 몇 차원의 벡터로 표현할지 (예: 128차원)
    # hidden_size: LSTM의 '기억 공간(은닉 상태)' 크기 (기본값 128)
    # num_layers: LSTM 층을 몇 개 쌓을지 (기본값 1)
    # dropout: 과적합을 방지하기 위해 얼마나 뉴런을 끌지 (기본값 0.1 = 10%)
    def __init__(self, vocab_size, embedding_dim, hidden_size=128, num_layers=1, dropout=0.1):
        # 부모 클래스(nn.Module)의 초기화 함수를 반드시 먼저 호출합니다.
        # (이유: 파이토치 모델로서 필요한 내부 설정들을 처리하기 위함입니다.)
        super(Encoder, self).__init__()

        # --- 1. 임베딩 레이어 정의 ---
        # 'nn.Embedding' 레이어를 'self.embedding'이라는 이름으로 생성합니다.
        # (vocab_size)개의 단어를 각각 (embedding_dim) 차원의 벡터로 매핑하는 '조회 테이블'을 만듭니다.
        # (이유: '1번 단어' 같은 숫자(ID)를 '의미를 가진 벡터'(예: 128차원)로 변환해야 모델이 이해할 수 있기 때문입니다.)
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # --- 2. LSTM 레이어 정의 ---
        # 'nn.LSTM' 레이어를 'self.lstm'이라는 이름으로 생성합니다.
        # (이유: LSTM은 문장의 순서(맥락)를 기억하며 읽어 내려가는 역할을 합니다.)
        self.lstm = nn.LSTM(
            # input_size: LSTM에 입력되는 각 시점(단어) 벡터의 차원. (임베딩 레이어의 출력 차원과 같아야 함)
            input_size=embedding_dim,
            # hidden_size: LSTM의 은닉 상태(h)와 셀 상태(c)의 크기. (모델의 '기억 용량'을 결정)
            hidden_size=hidden_size,
            # num_layers: 쌓아 올릴 LSTM 층의 수. (깊어질수록 복잡한 패턴 학습 가능)
            num_layers=num_layers,
            # batch_first=True: 입력 텐서의 차원 순서를 [batch_size, seq_len, input_size]로 설정합니다.
            # (이유: 이 옵션이 없으면 [seq_len, batch_size, input_size]가 되어 다루기 까다롭습니다. True로 설정하는 것이 직관적입니다.)
            batch_first=True,
        )
        # --- 3. 드롭아웃 레이어 정의 ---
        # 'nn.Dropout' 레이어를 'self.dropout'이라는 이름으로 생성합니다.
        # (이유: 훈련 중에 'dropout' 비율(예: 10%)만큼 임의의 뉴런을 0으로 만들어,
        # 모델이 특정 뉴런에 과도하게 의존하는 것을 방지합니다. (과적합 방지용))
        self.dropout=nn.Dropout(dropout)
        # hidden_size 값을 'self.hidden_size' 변수에 저장합니다.
        # (이유: 클래스 내부의 다른 함수나 외부에서 이 값을 참조해야 할 때 유용합니다.)
        self.hidden_size=hidden_size

    # [전체 흐름 2: 인코더 순전파(Forward Pass) 함수 정의]
    # 인코더 객체에 실제 데이터(src)가 입력되었을 때 실행되는 '흐름'을 정의합니다.
    # src: 소스(Source) 문장의 토큰 ID 텐서. (모양: [batch_size, seq_len])
    def forward(self, src):
        # --- 1. 임베딩 + 드롭아웃 ---
        # (1) self.embedding(src):
        #     입력 'src' (모양: [B, S])를 임베딩 레이어에 통과시켜 '밀집 벡터'로 변환합니다.
        #     (모양 변화: [B, S] -> [B, S, E]) (B:배치, S:시퀀스 길이, E:임베딩 차원)
        # (2) self.dropout(...):
        #     (1)의 결과인 임베딩 벡터에 드롭아웃을 적용합니다. (훈련 모드에서만 작동)
        # 'embedded' 텐서의 모양: [B, S, E]
        embedded = self.dropout(self.embedding(src))

        # --- 2. LSTM 순전파 ---
        # 'embedded' (단어 벡터 시퀀스)를 'self.lstm'에 입력합니다.
        # LSTM은 두 가지를 반환합니다: (모든 시점의 출력, (마지막 시점의 은닉 상태, 마지막 시점의 셀 상태))
        # (주석) 히든 상태와 같이 출력
        # (1) outputs:
        #     LSTM의 '마지막 레이어'에서 나온 '모든 시점(단어)'의 은닉 상태(h)를 모아놓은 것.
        #     (모양: [B, S, H]) (H: hidden_size)
        #     (이유: '어텐션'은 디코더가 '모든' 입력 단어의 정보를 참고해야 하므로, 이 'outputs'가 반드시 필요합니다.)
        # (2) (h, c):
        #     '모든 레이어'의 '마지막 시점'의 은닉 상태(h)와 셀 상태(c).
        #     h의 모양: [num_layers, B, H]
        #     c의 모양: [num_layers, B, H]
        #     (이유: 이 (h, c)가 문장 전체의 '문맥(Context)'을 압축한 벡터이며, 디코더의 '첫 번째 상태'로 전달되어 "이 문맥을 바탕으로 답변을 시작해라"는 신호를 줍니다.)
        outputs, (h, c) = self.lstm(embedded)

        # --- 3. 결과 반환 ---
        # (outputs: 어텐션용), (h, c: 디코더 초기화용) 세 가지 값을 모두 반환합니다.
        return outputs, h, c


# [전체 흐름 3: 인코더 생성 및 테스트]
# (이 부분은 모델이 잘 작동하는지 확인하기 위한 테스트 코드입니다.)

# (주석) 인코더 확인
# 'Encoder' 클래스(설계도)를 바탕으로 'encoder'라는 실제 모델 객체를 생성합니다.
# __init__ 함수가 호출됩니다.
# vocab_size=100 (단어 100개짜리 사전 가정)
# embedding_dim=32 (단어를 32차원으로 표현)
# num_layers=3 (LSTM을 3층으로 쌓음)
# (hidden_size는 기본값 128을 사용합니다.)
encoder = Encoder(vocab_size=100, embedding_dim=32, num_layers=3)

# 모델에 입력할 임의의 테스트 텐서(문장)를 생성합니다.
# [1, 2, 3, 4]는 '1번, 2번, 3번, 4번 단어'로 이루어진 문장 1개를 의미합니다.
# 현재 'input'의 모양(shape)은 [4] 입니다. (시퀀스 길이 4)
input=torch.tensor([1,2,3,4]) # 임의의 토큰

# [지금 하는 일: 모델 실행 및 결과 확인]
# 'encoder'의 'forward' 함수를 호출하여 'input'을 입력합니다.
# (중요!) 'input.unsqueeze(0)'를 사용하는 이유:
#     'self.lstm'을 'batch_first=True'로 설정했기 때문에,
#     모델은 항상 [batch_size, seq_len] 형태의 2D 텐서를 기대합니다.
#     'input'의 모양은 [4] (seq_len)이므로, 'unsqueeze(0)'를 사용해
#     맨 앞에 '배치 차원(batch_size=1)'을 강제로 추가해줍니다.
#     (모양 변화: [4] -> [1, 4])
# 'enc_out'에는 'outputs'가, 'h'와 'c'에는 각각 (h, c)가 저장됩니다.
enc_out, h, c = encoder(input.unsqueeze(0))

# (이전 코드...)
# encoder = Encoder(vocab_size=100, embedding_dim=32, num_layers=3)
# input=torch.tensor([1,2,3,4]) # 임의의 토큰
# enc_out, h, c = encoder(input.unsqueeze(0))
# (이전 코드 끝...)


# 반환된 'outputs' (여기서는 enc_out)의 모양을 출력합니다.
# (예상 모양: [batch_size, seq_len, hidden_size] -> [1, 4, 128])
# (이유: (B=1, S=4) 문장이 들어가서, (H=128) 크기의 '중간 생각'이 4개 나왔다는 의미)
#
# [여기서 B, S, H는 다음을 의미합니다]
#
# B = batch_size (배치 크기)
#   (이유: 모델을 훈련시킬 때 데이터를 '묶음'(배치) 단위로 처리하기 때문입니다.)
#   (예: 64개 문장을 한 번에 처리하면 B=64 입니다.)
#   (테스트 코드 'input.unsqueeze(0)'는 '1개'의 문장으로 '묶음'을 만들었으므로 B=1 입니다.)
#
# S = seq_len (시퀀스 길이)
#   (이유: 입력된 문장이 몇 개의 '토큰(단어)'으로 이루어져 있는지를 나타냅니다.)
#   (테스트 코드 'input=torch.tensor([1,2,3,4])'는 4개의 토큰으로 이루어져 있으므로 S=4 입니다.)
#
# H = hidden_size (은닉 상태 크기)
#   (이유: LSTM이 각 단어(시점)의 '정보/맥락'을 몇 차원의 벡터로 '요약'하는지를 나타냅니다.)
#   (이것은 모델의 '기억 용량' 또는 '생각의 복잡도'를 의미합니다.)
#   (테스트 코드 'Encoder(...)'는 기본값 hidden_size=128을 사용했으므로 H=128 입니다.)
#   (참고: 만약 nn.LSTM(..., bidirectional=True)를 썼다면 이 부분(outputs)은 H*2 가 됩니다.)
#
print(f'인코더 출력 결과: {enc_out.shape}')

# 반환된 'h' (최종 은닉 상태)의 모양을 출력합니다.
# (예상 모양: [num_layers, batch_size, hidden_size] -> [3, 1, 128])
# (이유: (L=3, B=1)에 대해 (H=128) 크기의 '최종 생각'이 나왔다는 의미)
#
# [여기서 L, B, H는 다음을 의미합니다]
#
# L = num_layers (레이어 수)
#   (이유: LSTM 층을 몇 개나 '쌓았는지'(stack)를 나타냅니다.)
#   (테스트 코드 'Encoder(..., num_layers=3)'에서 3으로 설정했으므로 L=3 입니다.)
#   (LSTM은 '모든 층'의 '마지막' 상태를 반환하므로, 이 차원이 존재합니다.)
#   (참고: 만약 bidirectional=True 였다면 이 차원은 L*2 가 됩니다.)
#
# B = batch_size (배치 크기)
#   (이유: 위와 동일합니다. 1개의 문장(B=1)에 대한 '최종 생각'입니다.)
#
# H = hidden_size (은닉 상태 크기)
#   (이유: 위와 동일합니다. '최종 생각'도 '중간 생각'과 동일한 '용량'(H=128)으로 요약됩니다.)
#
print(f'인코더 히든 상태: {h.shape}')

인코더 출력 결과: torch.Size([1, 4, 128])
인코더 히든 상태: torch.Size([3, 1, 128])


#### 디코더 구성

디코더는 생성 하고자 하는 문장의 직적 토큰으로 부터 다음 토큰을 생성하는 역할을 합니다.

이때 인코더가 만들어낸 입력 텍스트의 순환특성과 함께 직전 토큰을 입력 받아 새로운 토큰을 만들어 냅니다.

기본적인 LSTM(또는 GRU) 구조만 가지고 있을땐 인코더의 히든 상태를 디코더의 토큰이 들어가는 LSTM이 이어 받아 사용합니다.

<center><img src="https://drive.google.com/uc?export=view&id=1-j4OMIdZoyYgy7UY4KCjgPyeBwkrczR0" width="400"/></center>


Attention 기법을 추가하면 디코더는 인코더의 모든 시퀀스 결과 값과 마지막 시퀀스의 히든 상태를 Bahdanau Attention하여 컨텍스트 벡터를 만들어냅니다.
만들어진 컨텍스트 벡터는 디코더의 입력 토큰 임베딩과 concat 하여 디코더 LSTM 구조에 입력합니다.

<center><img src="https://drive.google.com/uc?export=view&id=1eVM1UMZBkK1k8F0m33dI44q35HSKBKzc" width="600"/></center>


In [ ]:
# [전체 흐름]
# 이 코드는 사용자가 요청한 두 번째 코드 블록, 즉 'Decoder' (디코더) 클래스를 정의하는 부분입니다.
# 'Encoder'가 질문(입력)을 읽고 '요약'하는 "독자"였다면,
# 'Decoder'는 그 요약본을 받아서 답변(출력)을 '한 단어씩' 써내려가는 "작가"입니다.

import torch
import torch.nn as nn
import torch.nn.functional as F

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size=128, num_layers=1, dropout=0.1):
        super(Decoder, self).__init__()

        # === 1. 기본 구성 요소 정의 ===

        # 🔤 임베딩 레이어: 단어 ID → 의미 벡터
        # 작동 원리: 정수로 된 단어 ID를 받아서 실수 벡터로 변환
        # 예: 단어 ID 5 → [0.12, -0.45, 0.78, 0.33, ...] (embedding_dim 개수의 실수들)
        # 왜 필요한가? 모델은 숫자만 이해할 수 있지만, 단어의 '의미'를 이해하려면 벡터 표현이 필요함
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # 🧠 LSTM 레이어: 시퀀스 데이터 처리하는 순환 신경망
        # input_size = embedding_dim + hidden_size 인 진짜 이유:
        #   - embedding_dim: '지금 보고 있는 단어'의 의미 (예: "사과"라는 단어 자체의 의미)
        #   - hidden_size: '질문 내용 중에서 지금 중요한 부분'의 의미 (예: "뭐야?"라는 질문의 핵심)
        #   → 두 정보를 합쳐서 "지금 이 단어를 보고 이 질문 맥락에서 다음에 무슨 단어를 써야 하지?" 판단
        self.lstm = nn.LSTM(
            input_size=embedding_dim + hidden_size,  # 입력 크기: 단어정보 + 질문맥락정보
            hidden_size=hidden_size,                 # LSTM이 한 번에 기억할 수 있는 정보량
            num_layers=num_layers,                   # LSTM을 몇 층 쌓을지 (층이 많을수록 복잡한 패턴 학습)
            batch_first=True                         # 입력 형태: [배치크기, 시퀀스길이, 특성크기]
        )

        # 🎯 최종 출력 레이어: 다음 단어 예측
        # 작동 원리: LSTM의 '생각 결과'를 받아서 '가능한 모든 다음 단어들'에 대한 점수 계산
        # 예: LSTM 출력 [0.1, 0.5, -0.2] → 단어점수 [2.1(안녕), -1.5(hello), 0.8(hi), ...]
        # 왜 필요한가? LSTM은 '의미'를 이해했지만, 실제로 '어떤 단어를 선택할지'는 이 레이어가 결정
        self.fc_out = nn.Linear(hidden_size, vocab_size)

        # 🛡️ 드롭아웃: 과적합 방지
        # 작동 원리: 훈련 중에 무작위로 일부 뉴런을 꺼서 모델이 특정 패턴에만 의존하지 않게 함
        # 예: 0.1 설정 = 10%의 연결을 임시로 끊음
        self.dropout = nn.Dropout(dropout)

        # === 2. Attention 메커니즘 (이 부분이 가장 중요!) ===

        # 🎯 어텐션 레이어 1: 디코더 상태 변환기
        # 하는 일: "디코더가 지금까지 생각한 내용"을 어텐션 계산에 맞게 변환
        # 구체적 예:
        #   입력: 디코더 상태 [0.1, 0.5, -0.2]
        #   출력: 변환된 상태 [0.3, -0.1, 0.4] (어텐션 계산에 더 적합한 형태)
        self.attention1 = nn.Linear(hidden_size, hidden_size)

        # 🎯 어텐션 레이어 2: 인코더 상태 변환기
        # 하는 일: "인코더가 분석한 질문의 모든 단어 정보"를 어텐션 계산에 맞게 변환
        # 구체적 예:
        #   입력: 인코더 출력들 [[0.1,0.2], [0.3,0.4], [0.5,0.6]] (3개 단어 정보)
        #   출력: 변환된 출력들 [[0.2,0.1], [0.4,0.3], [0.6,0.5]] (어텐션 계산에 더 적합한 형태)
        self.attention2 = nn.Linear(hidden_size, hidden_size)

        # 🎯 어텐션 레이어 3: 최종 점수 계산기
        # 하는 일: "이 인코더 단어가 얼마나 중요한지"를 하나의 숫자로 요약
        # 구체적 예:
        #   입력: 변환된 정보 [0.7, 0.8, 0.9] (복잡한 비교 결과)
        #   출력: 2.1 (단순한 중요도 점수)
        # bias=False인 이유: Bahdanau 논문에서 이렇게 하면 더 잘 작동한다고 제안함
        self.v = nn.Linear(hidden_size, 1, bias=False)

        self.hidden_size = hidden_size

    def forward(self, input_token, hidden, cell, encoder_outputs):
        """
        🔥 디코더의 한 단계 실행: 딱 한 단어만 생성하는 과정 🔥

        이 함수가 호출될 때마다:
        - 입력: 현재 단어 1개 + 지금까지의 기억
        - 출력: 다음 단어 1개에 대한 예측 + 갱신된 기억

        전체 문장을 생성하려면 이 함수를 여러 번 반복 호출해야 함
        """

        # =============================================
        # 🎯 1단계: ATTENTION 계산 - "질문 중에서 뭐가 중요한지 찾기"
        # =============================================

        # 📌 1-1: 디코더의 현재 상태 추출
        # hidden shape: [num_layers, batch_size, hidden_size]
        # 예: [3, 64, 256] → 3층 LSTM, 64개 문장, 각각 256차원 기억
        # [-1]은 "마지막 층의 상태만 사용" 의미 (가장 정제된 정보)
        dec_hidden = hidden[-1]  # shape: [batch_size, hidden_size]

        # 📌 1-2: 차원 확장 (브로드캐스팅 준비)
        # [batch_size, hidden_size] → [batch_size, 1, hidden_size]
        # 왜? encoder_outputs와 모양을 맞추기 위해 가짜 시퀀스 차원(1) 추가
        dec_hidden = dec_hidden.unsqueeze(1)

        # 📌 1-3: 어텐션 스코어 계산 (Bahdanau Attention 공식 구현)
        # 공식: score = v * tanh(W1 * h_dec + W2 * h_enc)

        # W1 * h_dec: 디코더 상태 변환
        dec_u = self.attention1(dec_hidden)  # shape: [batch_size, 1, hidden_size]

        # W2 * h_enc: 인코더 상태 변환
        enc_u = self.attention2(encoder_outputs)  # shape: [batch_size, src_seq_len, hidden_size]

        # W1*h_dec + W2*h_enc: 두 정보 결합 (브로드캐스팅 발생)
        # dec_u: [64, 1, 256] → [64, 10, 256] (자동 복사)
        # enc_u: [64, 10, 256] → [64, 10, 256] (그대로)
        concat_inputs = dec_u + enc_u  # shape: [batch_size, src_seq_len, hidden_size]

        # v * tanh(...): 최종 어텐션 스코어 계산
        score = self.v(torch.tanh(concat_inputs))  # shape: [batch_size, src_seq_len, 1]

        # 📌 1-4: 어텐션 가중치 계산 (소프트맥스)
        # score: [2.1, 0.5, -1.2] → a: [0.7, 0.2, 0.1] (확률로 변환, 합=1)
        a = F.softmax(score, dim=1)  # shape: [batch_size, src_seq_len, 1]

        # =============================================
        # 🎯 2단계: CONTEXT VECTOR 생성 - "중요한 정보만 요약하기"
        # =============================================

        # 📌 2-1: 가중합 계산 (어텐션 가중치 적용)
        # a * encoder_outputs: 각 인코더 단어에 중요도 가중치 곱하기
        # 예:
        #   a = [0.7, 0.2, 0.1] (세 단어의 중요도)
        #   encoder_outputs = [[단어1정보], [단어2정보], [단어3정보]]
        #   결과 = [0.7*단어1정보, 0.2*단어2정보, 0.1*단어3정보]
        weighted_encoder_outputs = a * encoder_outputs

        # 📌 2-2: 요약본 생성 (가중합 합치기)
        # torch.sum(..., dim=1): 시퀀스 차원을 따라 모두 더해서 하나의 벡터로
        # 결과: "가장 중요한 정보들만 모은 요약본"
        context = torch.sum(weighted_encoder_outputs, dim=1)  # shape: [batch_size, hidden_size]

        # =============================================
        # 🎯 3단계: LSTM 입력 준비 - "현재 정보 + 맥락 정보 합치기"
        # =============================================

        # 📌 3-1: 현재 입력 단어 처리
        input_token = input_token.unsqueeze(1)  # [batch_size] → [batch_size, 1]
        embedded = self.embedding(input_token)  # [batch_size, 1, embedding_dim]

        # 📌 3-2: 컨텍스트 벡터 차원 맞추기
        context = context.unsqueeze(1)  # [batch_size, hidden_size] → [batch_size, 1, hidden_size]

        # 📌 3-3: 두 정보 결합 (Concatenate)
        # embedded: [64, 1, 128] (현재 단어 의미)
        # context: [64, 1, 256] (질문 맥락 정보)
        # dec_input: [64, 1, 384] (두 정보를 옆으로 붙임)
        dec_input = torch.cat((embedded, context), dim=2)

        # =============================================
        # 🎯 4단계: LSTM 실행 - "다음 상태 계산하기"
        # =============================================

        # 📌 4-1: LSTM 한 스텝 실행
        # 입력:
        #   - dec_input: [batch_size, 1, embedding_dim + hidden_size] (현재 정보)
        #   - (hidden, cell): [num_layers, batch_size, hidden_size] (과거 기억)
        # 출력:
        #   - out: [batch_size, 1, hidden_size] (현재 스텝의 생각 결과)
        #   - (hidden, cell): [num_layers, batch_size, hidden_size] (갱신된 기억)
        out, (hidden, cell) = self.lstm(dec_input, (hidden, cell))

        # 📌 4-2: 출력 형태 정리
        out = out.squeeze(1)  # [batch_size, 1, hidden_size] → [batch_size, hidden_size]
        out = self.dropout(out)  # 과적합 방지를 위한 드롭아웃

        # =============================================
        # 🎯 5단계: 다음 단어 예측 - "이제 무슨 단어를 쓸지 결정"
        # =============================================

        # 📌 5-1: 단어 점수 계산
        # out: [batch_size, hidden_size] (LSTM의 생각)
        # output: [batch_size, vocab_size] (각 단어별 점수)
        output = self.fc_out(out)

        # 📌 5-2: 결과 반환
        return output, hidden, cell

# [전체 흐름 3: 디코더 생성 및 테스트]
# (주석) 디코더 확인
# (주의: 이 코드를 실행하려면, '이전 셀'의 'Encoder' 코드가 실행되어
# 'h', 'c', 'enc_out'이라는 변수가 메모리에 '미리' 정의되어 있어야 합니다.
# (h, c, enc_out = encoder(input.unsqueeze(0)) 부분이 실행되었어야 함))

# 'Decoder' 클래스(설계도)를 바탕으로 'decoder' 실제 모델 객체를 생성합니다.
# __init__ 함수가 호출됩니다.
# (Encoder 테스트와 동일한 값으로 설정하여 차원이 맞도록 합니다.)
decoder = Decoder(vocab_size=100,embedding_dim=32, num_layers=3)

# 디코더의 '첫 번째' 입력으로 사용할 '가짜' 토큰을 생성합니다.
# '2'는 'spm_command'에서 '--bos_id=2'로 설정한 '[BOS]' (문장 시작) 토큰 ID를 의미합니다.
# torch.tensor([2])의 모양은 [1]이며, 이는 'batch_size=1'을 의미합니다.
# (실제 훈련 시에는 'tar[:, 0]' (정답 배치의 0번째)을 사용합니다.)
input = torch.tensor([2]) # 임의의 토큰

# [지금 하는 일: 디코더 1스텝 실행 테스트]
# 'decoder'의 'forward' 함수를 'h', 'c', 'enc_out'과 함께 호출합니다.
# (1) input: [BOS] 토큰 (모양 [1])
# (2) h: 'Encoder'가 반환한 '최종 은닉 상태' (모양 [3, 1, 128]) (이전 셀에서 생성됨)
# (3) c: 'Encoder'가 반환한 '최종 셀 상태' (모양 [3, 1, 128]) (이전 셀에서 생성됨)
# (4) enc_out: 'Encoder'가 반환한 '모든 시점 출력' (모양 [1, 4, 128]) (이전 셀에서 생성됨)
#
# (반환값) 'output' (첫 번째 예측 단어 점수표), 'h', 'c' (디코더가 1스텝 갱신한 상태)
output, h, c  = decoder(input, h, c, enc_out)

# 디코더의 '최종 출력' (다음 단어 예측 점수표)의 '모양(shape)'을 출력합니다.
# (예상 모양: [batch_size, vocab_size] -> [1, 100])
# (이유: B=1, vocab_size=100 이므로)
print(f'디코더 출력 결과: {output.shape}')
# 디코더가 '1스텝 갱신한' 은닉 상태 'h'의 '모양(shape)'을 출력합니다.
# (예상 모양: [num_layers, batch_size, hidden_size] -> [3, 1, 128])
# (이유: L=3, B=1, H=128 이므로)
print(f'디코더 히든 상태: {h.shape}')

인코더 출력 결과: torch.Size([1, 100])
인코더 히든 상태: torch.Size([3, 1, 128])


In [ ]:
# [전체 흐름]
# 이 코드는 Seq2seq (질의응답) 모델 훈련에 필요한 '데이터'를 준비하는 과정입니다.
# 1. (1부: 데이터 탐색) 'train.csv' 파일을 읽어와서 'HS01'(질문), 'SS01'(답변) 열을 확인합니다.
# 2. (2부: SPDataSet 클래스 정의) 파이토치 'Dataset'을 상속받아, Seq2seq 모델에 맞게 데이터를 변환하는 'SPDataSet' 클래스를 설계합니다.
#    - 이 클래스는 'i'번째 데이터를 요청받으면( __getitem__ ),
#    - 'i'번째 질문(HS01)과 답변(SS01)을 토크나이저(sp)로 '숫자 리스트'로 변환합니다.
#    - 질문(inp)은 그대로 패딩(zero_pad)합니다.
#    - 답변(tar)은 '앞뒤'에 [BOS] (시작) 토큰과 [EOS] (끝) 토큰을 '붙인 후' 패딩합니다.
#    - 이렇게 가공된 (inp, tar) 텐서 쌍을 반환합니다.
# 3. (3부: DataLoader 생성 및 확인) 'SPDataSet'을 실제로 만들고, 'DataLoader'로 감싸서
#    '배치(batch)' 단위로 데이터를 잘 꺼내오는지 1개 배치만 테스트 출력합니다.

# pandas 라이브러리를 'pd'라는 별칭으로 가져옵니다.
# (이유: CSV 파일과 같은 표 형태의 데이터를 읽고 처리(DataFrame)하기 위해 필요합니다.)
import pandas as pd

# 'train.csv' 파일을 읽어와서 'df'라는 이름의 DataFrame 변수에 저장합니다.
# (이유: 파일에 어떤 데이터가 있는지 확인(탐색)하기 위한 코드입니다.)
df = pd.read_csv(f'train.csv')
# (주석) HS01: 질의(질문), SS01: 응답(답변)
# 'df' DataFrame에서 'HS01' 열과 'SS01' 열만 선택하여 출력합니다.
# (이유: Seq2seq 모델의 입력(질문)과 정답(답변)으로 사용할 데이터를 확인합니다.)
df[['HS01','SS01']]

,HS01,SS01
0,일은 왜 해도 해도 끝이 없을까? 화가 난다.,많이 힘드시겠어요. 주위에 의논할 상대가 있나요?
1,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.,급여가 줄어 속상하시겠어요. 월급이 줄어든 것을 어떻게 보완하실 건가요?
2,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...,회사 동료 때문에 스트레스를 많이 받는 것 같아요. 문제 해결을 위해 어떤 노력을 ...
3,직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...,관련 없는 심부름을 모두 하게 되어서 노여우시군요. 어떤 것이 상황을 나아질 수 있...
4,얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.,무시하는 것 같은 태도에 화가 나셨군요. 상대방의 어떤 행동이 그런 감정을 유발하는...
...,...,...
51623,나이가 먹고 이제 돈도 못 벌어 오니까 어떻게 살아가야 할지 막막해. 능력도 없고.,경제적인 문제 때문에 막막하시군요. 마음이 편치 않으시겠어요.
51624,몸이 많이 약해졌나 봐. 이제 전과 같이 일하지 못할 것 같아 너무 짜증 나.,건강에 대한 어려움 때문에 기분이 좋지 않으시군요. 속상하시겠어요.
51625,이제 어떻게 해야 할지 모르겠어. 남편도 그렇고 노후 준비도 안 되어서 미래가 걱정돼.,노후 준비에 대한 어려움 때문에 걱정이 많으시겠어요.
51626,몇십 년을 함께 살았던 남편과 이혼했어. 그동안의 세월에 배신감을 느끼고 너무 화가 나.,가족과의 문제 때문에 속상하시겠어요.


In [ ]:
# [전체 흐름 2: Seq2seq용 데이터셋 클래스 설계]
# 'SPDataSet'이라는 이름의 새 클래스를 정의합니다. 파이토치의 'Dataset' 클래스를 상속받습니다.
# (이유: Seq2seq 모델에 맞는 '입력(질문)'과 '정답(답변)' 텐서 쌍을 반환하는 맞춤형 데이터 변환기를 만들기 위함입니다.)
class SPDataSet(Dataset):
    # 'SPDataSet' 클래스의 초기화(생성자) 함수입니다. 객체가 생성될 때(예: SPDataSet(...)) 호출됩니다.
    # sp: 훈련된 SentencePiece 토크나이저 객체
    # max_len: 문장의 최대 길이를 몇으로 고정할지 (패딩/절단 기준)
    def __init__(self, sp, max_len):
        # 입력받은 max_len 값을 'self.max_len'이라는 클래스 변수에 저장합니다.
        # (이유: 클래스 내의 다른 함수(zero_pad, __getitem__)에서도 이 값을 사용해야 하기 때문입니다.)
        self.max_len = max_len
        # 'train.csv' 파일을 pandas로 즉시 읽어옵니다.
        # (이유: 이 클래스가 생성되는 시점에, 필요한 'HS01'(질문)과 'SS01'(답변) 데이터만 DataFrame으로 불러와 'self.df'에 저장해 둡니다.)
        self.df = pd.read_csv(f'train.csv')[['HS01','SS01']]
        # 입력받은 SentencePiece 객체를 'self.sp' 변수에 저장합니다.
        # (이유: __getitem__ 함수에서 문장을 토큰화할 때 이 객체를 사용해야 합니다.)
        self.sp = sp

    # [지금 하는 일: 패딩 함수 정의]
    # 'tok' (토큰화된 숫자 리스트)를 'self.max_len' 길이에 맞게 '패딩(padding)'하거나 '절단(truncating)'하는 함수입니다.
    # (이유: 모델에 데이터를 '배치' 단위로 넣으려면, 모든 문장의 길이를 동일하게(max_len) 맞춰야 합니다.)
    def zero_pad(self, tok):
        # 만약 토큰 리스트(tok)의 길이가 우리가 정한 최대 길이(max_len)보다 길거나 같다면,
        if len(tok) >= self.max_len:
            # 최대 길이(max_len)까지만 잘라내어 반환합니다. (예: max_len=60이면 0~59번 인덱스까지만)
            return tok[:self.max_len]
        # 만약 토큰 리스트의 길이가 최대 길이보다 짧다면,
        else:
            # 'np.zeros'를 사용해 'self.max_len' 길이만큼 0으로 채워진 numpy 배열을 만듭니다.
            # (이유: 여기서 0은 '[PAD]' 토큰 ID를 의미하며, 문장의 빈 공간을 채웁니다.)
            padding = np.zeros(self.max_len)
            # 이 0 배열의 '앞부분'(0번 인덱스부터 tok의 길이만큼)을 실제 토큰 리스트(tok) 내용으로 덮어씁니다.
            # (예: max_len=10, tok=[2,5,3] -> padding은 [2., 5., 3., 0., 0., 0., 0., 0., 0., 0.] 가 됨)
            padding[:len(tok)] = tok
            # 0으로 채워진(패딩된) 배열을 반환합니다.
            return padding

    # [지금 하는 일: 데이터 총 개수 반환 함수 정의]
    # Dataset 클래스의 필수 함수. 데이터셋에 총 몇 개의 데이터가 있는지 반환합니다.
    def __len__(self):
        # 'self.df' (DataFrame)의 전체 행(row)의 개수(len)를 반환합니다.
        return (len(self.df))

    # [지금 하는 일: 데이터 반환 함수 정의]
    # Dataset 클래스의 필수 핵심 함수. DataLoader가 'i'번째 데이터를 요청할 때 호출됩니다.
    # i: 0, 1, 2, ... 처럼 데이터의 인덱스(순번)
    def __getitem__(self, i):
        # 'self.df' (DataFrame)에서 'i'번째 행('iloc[i]')을 가져옵니다.
        # (결과: 'sent'는 'HS01'과 'SS01' 열을 가진 pandas Series 객체가 됩니다.)
        sent = self.df.iloc[i]
        # 'i'번째 행의 'HS01'(질문) 텍스트를 꺼내 'self.sp'(토크나이저)를 사용해 '토큰 ID 리스트'로 변환합니다.
        # (예: "안녕" -> [15, 10])
        sent1 = self.sp.encode_as_ids(sent['HS01'])
        # 'i'번째 행의 'SS01'(답변) 텍스트를 꺼내 'self.sp'(토크나이저)를 사용해 '토큰 ID 리스트'로 변환합니다.
        sent2 = self.sp.encode_as_ids(sent['SS01'])

        # (주석) 질의는 단어 토큰만 활용
        # 'sent1'(질문 토큰 리스트)을 'self.zero_pad' 함수에 넣어 'max_len' 길이에 맞게 패딩/절단합니다.
        # (이유: 이것이 Encoder의 '입력(input)'이 됩니다.)
        inp = self.zero_pad(sent1)
        # (주석) 응답에는 시작 토큰과 끝 토큰을 추가하여 문장의 시작과 끝을 알림
        # 'sent2'(답변 토큰 리스트)의 '앞'에는 [BOS] 토큰 ID (예: 2)를, '뒤'에는 [EOS] 토큰 ID (예: 3)를 '붙입니다'.
        # (예: [5, 8] -> [2, 5, 8, 3])
        # 그리고 이 합쳐진 리스트를 'self.zero_pad' 함수로 패딩/절단합니다.
        # (이유: 이것이 Decoder의 '정답(target)'이 됩니다. Decoder는 [BOS]를 보고 생성을 시작하고 [EOS]를 생성하면 멈추도록 학습해야 합니다.)
        tar = self.zero_pad([self.sp.bos_id()] + sent2 + [self.sp.eos_id()])

        # 최종적으로 두 가지를 반환합니다:
        # 1. torch.Tensor(inp): 패딩된 '질문' numpy 배열을 파이토치 '텐서'로 변환합니다. (모델의 입력값)
        # 2. torch.Tensor(tar): [BOS/EOS]가 추가되고 패딩된 '답변' numpy 배열을 파이토치 '텐서'로 변환합니다. (모델의 정답값)
        return torch.Tensor(inp), torch.Tensor(tar)


# [전체 흐름 3: 데이터셋 및 로더 생성]
# SentencePieceProcessor 객체를 생성(초기화)합니다.
# (이유: 이전에 'bpe' 폴더에 훈련시켜 저장해 둔 'spm_krsent.model' 파일을 불러와, 'sp'라는 이름의 토크나이저로 사용합니다.)
sp = spm.SentencePieceProcessor(model_file=f'./bpe/spm_krsent.model')
# 위에서 설계한 'SPDataSet' 클래스(설계도)를 바탕으로 실제 'dataset' 객체를 생성합니다.
# (이유: __init__ 함수가 호출되며, 'sp'(토크나이저)와 'max_len=60'을 인자로 전달합니다.)
dataset = SPDataSet(sp, 60)
# 'dataset' 객체를 'DataLoader'로 감싸 'dataloader'라는 이름의 '데이터 공급기'를 생성합니다.
# (이유: 'dataloader'는 훈련 시 'dataset'에서 데이터를 꺼내올 때,
# 1. batch_size=8: 8개씩 묶어서 '배치'로 만들어줍니다.
# 2. shuffle=True: 데이터를 모델에 넣기 전에 매 에포크마다 순서를 무작위로 섞어줍니다. (과적합 방지에 중요))
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# [지금 하는 일: 데이터 로더 작동 확인]
# 'dataloader'를 'for' 반복문으로 실행하여, 1개의 '배치'를 꺼내옵니다.
# (이유: 'dataloader'가 'SPDataSet'의 __getitem__을 8번(batch_size=8) 호출하고,
# 그 결과(inp 8개, tar 8개)를 하나로 묶어 'inp'와 'tar' 텐서에 할당합니다.)
# (inp의 모양: [8, 60], tar의 모양: [8, 60])
for inp, tar in dataloader:
    # 'inp' (8개의 질문 텐서 묶음)에서 [0] (첫 번째 질문)을 선택합니다.
    # '.long()'을 적용하는 이유: __getitem__에서 반환한 'torch.Tensor'(기본 float 타입)를 모델 입력에 적합한 '정수(long)' 타입으로 변환합니다.
    # (출력 예: [ 15, 10, 4, 0, 0, ...])
    print(f'질문 토큰 : {inp.long()[0]}')
    # 'tar' (8개의 답변 텐서 묶음)에서 [0] (첫 번째 답변)을 선택하고, '정수(long)' 타입으로 변환하여 출력합니다.
    # (출력 예: [ 2, 5, 8, 3, 0, ...]) (2=[BOS], 3=[EOS], 0=[PAD])
    print(f'응답 토큰 : {tar.long()[0]}')
    # 'for' 반복문을 즉시 중단합니다.
    # (이유: 데이터 로더가 잘 작동하는지 확인하기 위해 '딱 1개의 배치'만 꺼내보고 테스트를 종료하기 위함입니다.)
    break

질문 토큰 : tensor([5232, 1139, 7077,  104,   34, 1096, 2603, 1019, 1179, 7135,  101,  287,
         391,   19,  424, 7045,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0])
응답 토큰 : tensor([   2,  887, 7077,  111, 1452,  156, 2046, 3303,   34, 5429, 7045,    3,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0])


#### 모델 학습

Seq2seq 모델은 구성한 Eencoder와 Decoder를 동시학습 합니다.


**학습 프로세스**   

    1. 입력 데이터인 질문 토큰을 인코더에 입력
    2. 인코더의 출력과 히든 상태, 셀 상태를 디코더의 초기 입력으로 설정
    3. 디코더는 타겟 데이터인 응답 토큰이 모두 생성될 때 까지 반복하여 토큰을 생성
    4. 디코더에 인코더 출력, 히든 상태, 셀 상태가 입력하여 얻은 디코더 출력 토큰, 히든 상태, 셀 상태를 다음 디코더 입력으로 설정하여 반복
    5. 매 반복마다 손실을 누적 연산하고 반복이 끝나면 역전파를 진행

In [ ]:
# [전체 흐름]
# 이 코드는 이전에 정의한 'Encoder'와 'Decoder' 모델을 '훈련시킬 준비'를 하는 과정입니다.
# 1. 'optim' (옵티마이저) 라이브러리를 가져옵니다.
# 2. 훈련에 필요한 '하이퍼파라미터'(모델 크기, 학습률, 배치 크기 등)를 설정합니다.
# 3. 이전에 훈련시킨 SentencePiece '토크나이저'를 불러옵니다.
# 4. 'Encoder'와 'Decoder' 모델의 '실제 객체'를 생성(초기화)합니다.
# 5. 'SPDataSet'과 'DataLoader'를 생성하여 훈련 데이터를 준비합니다.
# 6. 훈련을 가속화하기 위해 'GPU(cuda)' 사용을 설정하고, '두 모델 모두' GPU로 이동시킵니다.
# 7. '손실 함수(Loss Function)'와 '옵티마이저(Optimizer)'를 정의합니다. (훈련의 핵심 도구)

# [지금 하는 일: 필요 라이브러리 임포트]
# 파이토치의 'optim' 모듈을 'optim'이라는 별칭으로 가져옵니다.
# (이유: 'Adam', 'SGD' 등 모델의 가중치를 업데이트(학습)하는 '옵티마이저'를 사용하기 위해 필요합니다.)
import torch.optim as optim

# [지금 하는 일: 하이퍼파라미터 설정]
# (주석) '하이퍼파라미터'란, 모델이 스스로 학습하는 값(가중치)이 아니라,
# (주석) '사람'이 직접 설정해줘야 하는 값들입니다. 이 값들에 따라 모델의 성능이 크게 달라집니다.

# embedding_dim: 단어를 표현하는 벡터의 차원(크기)을 128로 설정합니다.
embedding_dim = 128
# max_len: 문장의 최대 길이를 60으로 설정합니다. (SPDataSet으로 전달될 값)
max_len = 60
# hidden_size: Encoder와 Decoder의 LSTM 은닉 상태(hidden state) 벡터 크기를 256으로 설정합니다.
hidden_size = 256
# num_layers: Encoder와 Decoder의 LSTM 레이어를 4개 층으로 쌓아 올리도록 설정합니다.
# (이유: Seq2seq에서는 인코더와 디코더의 층 수를 동일하게 맞춰야, (h, c) 상태를 원활하게 전달할 수 있습니다.)
num_layers = 4
# learning_rate (학습률): 옵티마이저가 가중치를 업데이트할 때의 '보폭(step size)'을 1e-4 (0.0001)로 설정합니다.
learning_rate = 1e-4
# num_epochs (에포크 수): 전체 훈련 데이터를 총 50번 반복해서 학습하도록 설정합니다.
num_epochs = 50
# batch_size: 훈련 데이터를 한 번에 64개씩 묶어서 모델에 넣고 학습시키도록 설정합니다.
batch_size = 64

# [지금 하는 일: 토크나이저 불러오기]
# 이전에 훈련시킨 'bpe/spm_krsent.model' 파일을 'SentencePieceProcessor'로 불러와 'sp' 객체로 만듭니다.
sp = spm.SentencePieceProcessor(model_file=f'./bpe/spm_krsent.model')
# 'sp' 토크나이저가 가지고 있는 전체 어휘(단어)의 개수를 'get_piece_size()' 함수로 가져와 'vocab_size' 변수에 저장합니다. (예: 8000)
# (이유: 이 'vocab_size'는 모델의 Embedding 레이어와 최종 출력 레이어의 크기를 결정하는 데 필수적입니다.)
vocab_size = sp.get_piece_size()

# [지금 하는 일: 모델 초기화]
# 'Encoder' 클래스(설계도)를 바탕으로 실제 'encoder' 객체를 생성합니다.
# (이유: __init__ 함수에 위에서 설정한 'vocab_size', 'embedding_dim', 'hidden_size', 'num_layers'를 전달하여
# 우리가 원하는 구조의 '인코더' 모델을 만듭니다.)
encoder = Encoder(vocab_size, embedding_dim, hidden_size, num_layers)
# 'Decoder' 클래스(설계도)를 바탕으로 실제 'decoder' 객체를 생성합니다.
# (이유: 디코더 역시 동일한 하이퍼파라미터로 설정하여, 인코더와 '모양'(차원)이 맞도록 합니다.)
decoder = Decoder(vocab_size, embedding_dim, hidden_size, num_layers)

# [지금 하는 일: 데이터셋 및 로더 생성]
# (주석) 간략한 실험을 위해 평가데이터 분리(random_split) 과정을 생략합니다.
# 'SPDataSet' 클래스(설계도)를 바탕으로 실제 'dataset' 객체를 생성합니다.
# (이유: 'sp'(토크나이저)와 'max_len'(60)을 전달하여, __getitem__이 (질문 텐서, 답변 텐서) 쌍을 반환하도록 합니다.)
dataset = SPDataSet(sp, max_len)
# 'dataset' 객체를 'DataLoader'로 감싸 'dataloader'라는 이름의 '데이터 공급기'를 생성합니다.
# (이유: 'batch_size'(64)만큼 데이터를 묶어주고, 'shuffle=True'로 매 에포크마다 데이터 순서를 섞어줍니다.)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
# 'len(dataset)' (SPDataSet의 __len__ 함수가 호출됨)을 통해
# 'dataset'에 포함된 총 데이터(질의응답 쌍)의 개수를 확인하고 출력합니다.
print(f'train dataset size: {len(dataset)}')

# [지금 하는 일: GPU(Device) 설정]
# 훈련을 실행할 장치(device)를 설정합니다.
# (이유: torch.cuda.is_available()이 True(GPU 사용 가능)이면 'cuda'를, 아니면 'cpu'를 'device' 변수에 저장합니다.)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 'encoder' 모델을 '.to(device)'를 사용해 설정된 장치(GPU 또는 CPU)로 보냅니다.
# (이유: 인코더의 모든 가중치와 연산이 GPU에서 수행되도록 하여 훈련 속도를 가속화합니다.)
encoder = encoder.to(device)
# 'decoder' 모델도 '.to(device)'를 사용해 '동일한' 장치(GPU 또는 CPU)로 보냅니다.
# (이유: 인코더와 디코더가 서로 텐서(h, c, outputs)를 주고받아야 하므로, 두 모델은 반드시 같은 장치에 있어야 합니다.)
decoder = decoder.to(device)

# [지금 하는 일: 손실 함수(Loss Function) 정의]
# 'nn.CrossEntropyLoss'를 'loss_fn'이라는 이름의 객체로 생성합니다.
# (이유: 이 함수는 디코더가 출력한 '다음 단어 예측 점수'(logit, [B, vocab_size])와
# '실제 정답 단어 ID'(label, [B])를 비교하여 '손실'을 계산합니다.)
# 'ignore_index=0' 옵션을 설정합니다.
# (이유: 정답(target) 텐서에서 값이 '0'인 위치는 '손실 계산에서 제외'하라는 의미입니다.
# 'SPDataSet'의 'zero_pad' 함수가 0을 [PAD] 토큰으로 사용했기 때문에,
# 모델이 '패딩' 부분을 맞추려고 애쓰지 않도록(즉, 실제 단어 부분만 학습하도록) 설정하는 것이 매우 중요합니다.)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

# [지금 하는 일: 옵티마이저(Optimizer) 정의]
# (주석) 두 파라미터 리스트를 합쳐서 단일 옵티마이저 생성
# 'encoder.parameters()'를 호출하여 '인코더'가 학습해야 할 모든 가중치(파라미터) 리스트를 가져옵니다.
encoder_params = list(encoder.parameters())
# 'decoder.parameters()'를 호출하여 '디코더'가 학습해야 할 모든 가중치 리스트를 가져옵니다.
decoder_params = list(decoder.parameters())
# 'torch.optim.Adam' 옵티마이저를 'optimizer'라는 이름의 객체로 생성합니다.
# (이유: 'Adam' 옵티마이저가 '인코더'와 '디코더'의 가중치를 '모두' 업데이트하도록,
# 'encoder_params + decoder_params' (두 리스트를 합친 하나의 리스트)를 학습 대상으로 전달합니다.
# 'lr=learning_rate' (0.0001)로 학습률을 설정합니다.)
# (이렇게 하면, 훈련 중 'loss.backward()'가 호출되었을 때,
# 'optimizer.step()' 한 번으로 Seq2seq 시스템 '전체'가 학습됩니다.)
optimizer = torch.optim.Adam(encoder_params + decoder_params, lr=learning_rate)

train dataset size: 51628


In [ ]:
# [전체 흐름 1: 훈련 루프(Epoch Loop) 시작]
# 'num_epochs' (예: 50)번 만큼 전체 데이터셋을 반복 학습하는 외부 루프를 시작합니다.
# (이유: 데이터셋을 한 번만 보는 것보다 여러 번 반복(epoch)해서 봐야 모델이 충분히 학습할 수 있습니다.)
# 'e' 변수에는 0, 1, 2, ..., 49 (현재 에포크 번호)가 차례대로 들어갑니다.
# (비유: "영어 문제집"을 1번 풀고(e=0), 2번 풀고(e=1)... 총 50번 푸는 과정입니다.)
for e in range(num_epochs):
    # [지금 하는 일: (1) 훈련 모드 설정]
    # 'encoder' 모델을 '훈련 모드'로 전환합니다.
    # (이유: 드롭아웃(Dropout) 같이 훈련 시에만 활성화되어야 하는 기능들을 켜기 위함입니다. 과적합 방지에 도움이 됩니다.)
    # (비유: 학생(모델)이 "이제부터 공부(훈련) 시작!" 모드로 자세를 바로잡는 것입니다.)
    encoder.train()
    # 'decoder' 모델도 '훈련 모드'로 전환합니다.
    # (이유: 인코더와 마찬가지로, 드롭아웃 등을 활성화하여 훈련을 진행합니다.)
    decoder.train()
    # '이번 에포크(e)'의 총 손실을 누적할 변수를 0.0으로 초기화합니다.
    # (이유: 배치(batch)마다 계산되는 손실(loss)을 여기에 계속 더해서, 나중에 에포크 전체의 평균 손실을 계산하기 위함입니다.)
    # (비유: "이번 문제집(에포크)에서 총 몇 점 틀렸는지" 기록할 '오답 점수 총합'을 0점으로 리셋합니다.)
    total_loss = 0.0

    # [전체 흐름 2: 배치 루프(Batch Loop) 시작]
    # 'dataloader'(데이터 공급기)에서 'batch_size'(예: 64)만큼 묶인 데이터 배치를 하나씩 순서대로 꺼내옵니다.
    # 'enumerate'를 사용하여 'i'에는 현재 배치의 인덱스(0, 1, 2...)를, '(inp, tar)'에는 실제 데이터 묶음을 받습니다.
    # (비유: "영어 문제집"을 '한 페이지'(batch)씩 뜯어서 푸는 과정입니다. 64문제가 한 페이지입니다.)
    # 'inp' (Input): 64개의 '질문' 텐서 묶음 (모양: [64, max_len]) (예: "How are you?")
    # 'tar' (Target): 64개의 '답변' 텐서 묶음 (모양: [64, max_len]) (예: "[BOS] I am fine [EOS]")
    for i, (inp, tar) in enumerate(dataloader):
        # [지금 하는 일: (2) 데이터 준비 (GPU 이동)]
        # 'inp' (질문 텐서)를 '.long()' (64비트 정수) 타입으로 변환합니다.
        # (이유: 'nn.Embedding' 레이어는 부동소수점(float)이 아닌 정수(long) 타입의 인덱스를 입력으로 받기 때문입니다.)
        # '.to(device)'를 사용해 이 텐서를 GPU(또는 CPU) 메모리로 보냅니다.
        # (이유: 모델(encoder)이 GPU에 있으므로, 입력 데이터도 같은 장치(device)로 보내야 연산이 가능합니다.)
        # (비유: 'CPU 책상'에 있던 '문제지(inp)'를 'GPU 작업대'로 옮깁니다.)
        inp = inp.long().to(device)
        # 'tar' (답변 텐서)도 '정수(long)' 타입으로 변환하고, 'device'(GPU)로 보냅니다.
        # (이유: 손실을 계산할 때 모델의 출력(GPU에 있음)과 정답(tar)을 비교해야 하므로, 둘 다 같은 장치에 있어야 합니다.)
        # (비유: 'CPU 책상'에 있던 '정답지(tar)'도 'GPU 작업대'로 옮깁니다.)
        tar = tar.long().to(device)

        # [지금 하는 일: (3) 기울기(Gradient) 초기화]
        # 'optimizer'(Adam)에 연결된 모든 파라미터(인코더와 디코더의 모든 가중치)들의 '기울기(gradient)' 값을 0으로 리셋(초기화)합니다.
        # (이유: 이 작업을 하지 않으면, '이전 배치(i-1)'에서 계산된 기울기가 '현재 배치(i)'의 기울기에 '누적'되어, 학습이 완전히 잘못된 방향으로 진행됩니다. 매 배치마다 '새로운 계산'을 위해 반드시 초기화해야 합니다.)
        # (비유: '이전 페이지(배치)'를 풀면서 "어떻게 고쳐야 할지" 적어둔 '오답노트(기울기)'를
        # '새 페이지(배치)'를 풀기 전에 '깨끗하게 지우는' 과정입니다.)
        optimizer.zero_grad()

        # ------ 인코더 순전파 ------
        # [지금 하는 일: (4) 인코더 순전파 실행]
        # (비유: 64명의 학생(배치)이 "How are you?"라는 '문제(inp)'를 '읽고 요약'하는 단계입니다.)
        # 준비된 'inp' (질문 배치, [64, 60])를 'encoder' 모델의 'forward' 함수에 입력합니다.
        # 인코더는 3개의 값을 반환합니다:
        # 1. 'enc_output': 인코더의 '모든 시점' 은닉 상태 출력. (모양: [64, 60, H])
        #    (이유: '어텐션'이 참고해야 할 원본 질문의 '모든 단어' 정보입니다.)
        #    (비유: "How", "are", "you" 각각에 대한 '중간 생각'들. (어텐션이 '참고서'로 사용))
        # 2. 'h': 인코더의 '마지막 시점' '은닉 상태'. (모양: [L, 64, H])
        # 3. 'c': 인코더의 '마지막 시점' '셀 상태'. (모양: [L, 64, H])
        #    (이유: (h, c)는 질문 문장 전체를 '요약'한 '문맥 벡터(Context Vector)'이며, 디코더의 '시작 상태'로 전달됩니다.)
        #    (비유: "How are you?"를 다 읽고 난 '최종 요약 생각'. (디코더의 '첫 번째 생각'으로 전달))
        enc_output, h, c = encoder(inp)

        # (주석) 디코더 초기 hidden을 인코더 hidden으로 설정

        # (주석) 디코더 첫 입력: <bos> 토큰 ([BOS])
        # [지금 하는 일: (5) 디코더 초기 입력 및 상태 설정]
        # (비유: 64명의 학생(디코더)이 "받아쓰기"를 시작할 준비를 합니다.)
        #
        # 디코더가 '답변 생성을 시작'하기 위한 '첫 번째 입력 토큰'을 설정합니다.
        # 'tar[:, 0]'는 'tar' (답변 배치, [64, 60])에서 '모든 배치'(:)의 '0번째 토큰'을 의미합니다.
        # (이유: 'SPDataSet'에서 답변(tar)을 만들 때 '[BOS]' (시작 토큰, 예: ID 2)를 맨 앞에 넣었으므로,
        # 'tar[:, 0]'는 64개 배치의 '[BOS]' 토큰 ID 리스트가 됩니다. (모양: [64]))
        # (비유: 선생님이 "자, 이제 받아쓰기 시작한다! '시작 신호([BOS])' 줄게!" -> 이 신호가 '첫 힌트'입니다.)
        dec_input = tar[:,0]
        # 디코더의 '첫 번째 은닉 상태'로 '인코더'가 반환한 '최종 은닉 상태(h)'를 설정합니다.
        # (이유: "인코더가 요약한 이 문맥(h)을 바탕으로 답변 생성을 시작해라"고 알려주는 것입니다.)
        # (비유: "방금 읽은 'How are you?' (h, c) 잊지 말고, 그거 생각하면서 써!")
        dec_hidden = h
        # 디코더의 '첫 번째 셀 상태'로 '인코더'가 반환한 '최종 셀 상태(c)'를 설정합니다.
        dec_cell = c
        # '이번 배치'의 '누적 손실'을 계산하기 위한 변수를 0.0으로 초기화합니다.
        # (이유: 디코더는 단어를 '한 스텝씩' 생성하며, 각 스텝마다의 'step_loss'가 발생합니다.
        # 이 'loss' 변수는 "I", "am", "fine" ... 각 단어에서 틀린 점수를 '전부 합산'하는 '총합' 변수입니다.)
        loss = 0.0

        # [전체 흐름 3: 디코더 스텝(단어) 루프 시작 (교사 강요)]
        # (비유: "받아쓰기"의 '가장 핵심'이 되는 부분입니다. '한 단어'씩 예측하고 '바로 채점'받는 과정입니다.)
        #
        # (주석) 교사 강요(teacher forcing) - 다음 입력으로 타겟을 피딩(feeding)
        # '답변' 텐서(tar)의 '두 번째 토큰'(인덱스 1)부터 '마지막 토큰'(max_len-1)까지, '한 단어(스텝)씩' 반복하는 루프를 시작합니다.
        # 'tar.size(1)'은 문장의 최대 길이(max_len=60)입니다.
        # 't' 변수에는 1, 2, 3, ..., 59 가 차례대로 들어갑니다.
        # (이유: 0번째([BOS])는 '입력'으로 썼으므로, '예측'은 1번째 단어부터 시작해야 합니다.
        # t=1 일 때: [BOS]를 보고 -> "I"를 예측함
        # t=2 일 때: "I"를 보고 -> "am"을 예측함
        # t=3 일 때: "am"을 보고 -> "fine"을 예측함 ...
        for t in range(1, tar.size(1)):
            # [지금 하는 일: (6) 디코더 1 스텝 순전파]
            # (비유: 학생(디코더)이 '이전 힌트'와 '문제 요약본'을 보고 '다음 1단어'를 예측합니다.)
            #
            # 'decoder' 모델의 'forward' 함수를 호출하여 '딱 한 단어'를 예측합니다.
            # [입력값]
            # 1. 'dec_input': '직전 스텝'의 토큰 (t=1일 때는 [BOS] 토큰, 모양: [64])
            # 2. 'dec_hidden', 'dec_cell': '직전 스텝'의 (은닉, 셀) 상태 (t=1일 때는 인코더의 (h, c))
            # 3. 'enc_output': '인코더'의 모든 시점 출력 (어텐션 계산용)
            # [반환값]
            # 1. 'pred': '이번 스텝(t)'에서 예측한 '다음 단어'의 점수 분포(logit). (모양: [64, vocab_size])
            #    (예: "I": 9.5점, "am": 2.1점, "fine": -1.0점 ...)
            # 2. 'dec_hidden', 'dec_cell': '이번 스텝(t)'에서 '갱신된' (은닉, 셀) 상태. (이 값은 다음 루프(t+1)의 입력으로 다시 들어감)
            pred, dec_hidden, dec_cell = decoder(dec_input, dec_hidden, dec_cell, enc_output)

            # [지금 하는 일: (7) 스텝 손실 계산]
            # (비유: 선생님이 학생의 예측을 '정답지'와 '즉시 비교 채점'합니다.)
            #
            # (주석) 정답 토큰과 손실
            # 'loss_fn'(CrossEntropyLoss)을 사용하여 '이번 스텝'의 손실을 계산합니다.
            # (1) 'pred': 모델의 '예측값' (모양: [64, vocab_size]) (예: "I" 9.5점...)
            # (2) 'tar[:, t]': '이번 스텝(t)'의 '실제 정답' 단어 ID (모양: [64]) (예: "I"의 ID 5번)
            # (결과: 'step_loss'에는 "5번 단어에 9.5점을 줬으니, 손실은 0.1점이다" 처럼 '틀린 정도'가 계산됩니다.)
            # (참고: 'loss_fn'은 'ignore_index=0'으로 설정되었기 때문에, 정답(tar[:, t])이 0([PAD])인 위치는
            # "아, 여긴 빈칸이네. 채점 안 함." 하고 알아서 무시하고 손실을 계산합니다.)
            step_loss = loss_fn(pred, tar[:, t])

            # (주석) 배치 전체가 패딩이면 손실이 nan 출력
            # 만약 'step_loss'가 'NaN'(Not a Number, 숫자가 아님)이 되었는지 확인합니다.
            # (이유: 'ignore_index=0'일 때, 'tar[:, t]'의 64개 값이 '모두' 0([PAD])이면,
            # "채점할 단어가 하나도 없네?"가 되어 손실 계산 결과가 'NaN'이 될 수 있습니다.
            # 이는 이 배치(64개 문장)가 '모두' 이 길이(t) 이전에 끝났다는 의미입니다.)
            if step_loss.isnan():
                # 손실이 NaN이면 (즉, 모든 배치가 패딩이면) 더 이상 이 문장에 대해 훈련할 필요가 없으므로,
                # 'for t ...' (디코더 스텝 루프)를 '즉시 중단(break)'합니다.
                # (비유: "64명 학생 전부 답안지 제출(EOS) 끝났네. 받아쓰기 종료.")
                break
            # (손실이 NaN이 아닐 경우) 유효한 'step_loss'를 'loss' 변수에 '누적'합니다.
            # (이유: "I" 틀린 점수 + "am" 틀린 점수 + "fine" 틀린 점수 ... 를 '전부 합산'합니다.)
            loss += step_loss

            # [지금 하는 일: (8) 교사 강요 실행]
            # (비유: 여기가 '교사 강요(Teacher Forcing)'의 '핵심'입니다.)
            #
            # (주석) Teacher forcing: 다음 시점 입력은 "현재 정답 토큰"
            # '다음 스텝(t+1)'의 디코더 '입력'('dec_input')으로,
            # 모델의 '예측'(pred)이 '아니라' '실제 정답'('tar[:, t]')을 사용하도록 설정합니다.
            #
            # (예시: t=1일 때)
            #   - 모델 예측(pred) = "am" (틀림)
            #   - 실제 정답(tar[:, 1]) = "I" (맞음)
            #   - [만약 교사 강요가 없다면]: 다음 힌트(dec_input)는 "am"이 됩니다.
            #     (-> 학생은 "am" 다음 단어("fine"?)를 예측하게 되어, 훈련이 '산으로 갑니다'.)
            #   - [교사 강요 (이 코드)]: 다음 힌트(dec_input)는 '정답'인 "I"가 됩니다!
            #     (-> 학생은 (비록 틀렸지만) "I" 다음 단어("am")를 예측하는 '올바른' 훈련을 이어갈 수 있습니다.)
            #
            # (이유: 훈련 초기에 멍청한 모델이 '한 번' 잘못 예측하면, 그 '잘못된 예측'을 힌트로
            # '연쇄적으로' 망하는 것을 방지하고, "일단 '올바른 길'로만 걸어가면서 '각 스텝마다
            # 예측하는 법'만이라도 제대로 배워라"고 '강제'하는 것입니다.)
            dec_input = tar[:, t]

        # [전체 흐름 4: 역전파 및 가중치 업데이트]
        # (디코더 스텝 루프(for t...)가 끝난 후, 즉 한 문장 생성이 끝난 후)
        # (비유: "받아쓰기 한 페이지(배치)"가 끝난 후, '최종 채점' 및 '오답노트 정리' 단계입니다.)
        #
        # (주석) 평균 손실 계산
        # 누적된 손실('loss')을 실제 진행된 스텝 수('t')로 나누어, 이 배치의 '단어 당 평균 손실'을 계산합니다.
        # (이유: 문장마다 유효한 단어 길이가 다르기 때문에(어떤 문장은 t=10에서, 어떤 문장은 t=59에서 끝남),
        # '총합(loss)'이 아닌 '평균'을 내야 "이번 배치가 평균적으로 얼마나 틀렸는지" 공평하게 계산할 수 있습니다.)
        batch_loss = loss / t

        # [지금 하는 일: (9) 역전파(Backpropagation)]
        # (비유: "틀린 문제(batch_loss)의 원인을 '거꾸로' 추적하라!")
        #
        # 계산된 'batch_loss'를 바탕으로, '손실에 영향을 미친 모든' 가중치들의 '기울기(gradient)'를 계산합니다.
        # '기울기'란 "손실을 줄이려면, 이 가중치를 ( + / - ) 방향으로 (얼마나) 수정해야 하는가?"를 나타내는 값입니다.
        # (이유: 이 과정은 'loss'가 계산된 '모든 스텝(t=1~59)'을 거슬러 올라가며(BPTT: Backpropagation Through Time),
        # '디코더'의 모든 가중치뿐만 아니라, 어텐션과 '인코더'의 가중치까지 "누가 얼마나 잘못했는지" 책임을 묻습니다.)
        batch_loss.backward()

        # [지금 하는 일: (10) 가중치 업데이트]
        # (비유: "분석 끝! '오답노트(기울기)'에 적힌 대로 '뇌(가중치)'를 수정하라!")
        #
        # 'optimizer'(Adam)가 '.backward()'로 계산된 '기울기(오답노트)'를 바탕으로,
        # 'encoder'와 'decoder'의 모든 가중치를 '실제로 업데이트'(수정)합니다.
        # (이것이 '학습'이 일어나고 모델이 "똑똑해지는" 실제 순간입니다.)
        optimizer.step()

        # [지금 하는 일: (11) 에포크 손실 누적]
        # 이번 배치의 평균 손실('batch_loss')에서 텐서 그래프를 제외한 순수 '값(.item())'만 추출하여,
        # 'total_loss'(에포크 총 손실) 변수에 더해줍니다.
        # (이유: 'total_loss'는 "이번 문제집(에포크)에서 총 몇 점 틀렸는지" 기록하는 변수이므로,
        # '이번 페이지(배치)'에서 틀린 점수(batch_loss)를 더해줍니다.)
        total_loss += batch_loss.item()

        # [지금 하는 일: (12) 중간 경과 출력]
        # '만약' 현재 배치 인덱스(i)에 1을 더한 값이 200의 배수(예: 200, 400, 600...)가 '될 때마다',
        # (이유: 훈련 과정이 너무 길 때(예: 배치가 10000개), 중간중간 "지금 잘 배우고 있나?"
        # (손실이 줄어드는지) 확인하기 위함입니다.)
        if (i+1) % 200 == 0:
            # 현재 에포크(e), 현재까지 진행된 배치 수(i+1),
            # 그리고 '현재까지의 평균 손실'(total_loss / (i+1))을 계산하여 화면에 출력합니다.
            # (비유: "5번 문제집(에포크) 푸는 중, 200페이지(배치)까지 풀었는데, 페이지당 평균 1.5점 틀렸네.")
            print(f"Epoch: {e}, Batch: {i+1}, Loss: {total_loss/(i+1)}")

    # [전체 흐름 5: 에포크 결과 출력]
    # (배치 루프(for i...)가 끝난 후, 즉 1 에포크(문제집 한 권)가 끝난 후)
    # 이번 에포크(e)의 '최종 평균 손실'(total_loss / (i+1))을 화면에 출력합니다.
    # ((i+1)은 최종적으로 총 배치 개수가 됩니다.)
    # (비유: "5번 문제집(에포크) 다 풀었다! 이번엔 페이지당 평균 1.2점 틀렸네. (지난번엔 1.5점이었는데 0.3점 향상!)")
    print(f"====>Epoch: {e}, Loss: {total_loss/(i+1)}")

Epoch: 0, Batch: 200, Loss: 6.34449214220047
Epoch: 0, Batch: 400, Loss: 5.871312553882599
Epoch: 0, Batch: 600, Loss: 5.700107701619467
Epoch: 0, Batch: 800, Loss: 5.603801011443138
====>Epoch: 0, Loss: 5.60002100777892
Epoch: 1, Batch: 200, Loss: 5.291472272872925
Epoch: 1, Batch: 400, Loss: 5.304937070608139
Epoch: 1, Batch: 600, Loss: 5.299239727656047
Epoch: 1, Batch: 800, Loss: 5.304503373503685
====>Epoch: 1, Loss: 5.304135610327549
Epoch: 2, Batch: 200, Loss: 5.171129469871521
Epoch: 2, Batch: 400, Loss: 5.030425117015839
Epoch: 2, Batch: 600, Loss: 4.9531835826238
Epoch: 2, Batch: 800, Loss: 4.927866156101227
====>Epoch: 2, Loss: 4.927523833963803
Epoch: 3, Batch: 200, Loss: 4.747872762680053
Epoch: 3, Batch: 400, Loss: 4.7323514592647555
Epoch: 3, Batch: 600, Loss: 4.718304278055827
Epoch: 3, Batch: 800, Loss: 4.696984837055206
====>Epoch: 3, Loss: 4.696343963474146
Epoch: 4, Batch: 200, Loss: 4.530886920690537
Epoch: 4, Batch: 400, Loss: 4.538710514903069
Epoch: 4, Batch: 60

> 손실이 계속 줄어들고 있으므로 충분히 긴 학습이 진행되어야 적절한 텍스트를 만드는 모델을 구성할 수 있습니다.

#### 텍스트 생성해보기

훈련된 Seq2seq에 질문을 인코더에 입력하고 시작토큰(bos)를 디코더의 첫 토큰으로 입력하여 텍스트를 생성해 봅니다.

In [ ]:
# [전체 흐름 1: 추론(Inference)용 데이터로더 생성]
# 'DataLoader'를 '추론(테스트)'용으로 새로 생성합니다.
# batch_size=1: 한 번에 '하나'의 (질문, 답변) 쌍만 꺼내오도록 설정합니다.
# (이유: '추론' 시에는 훈련과 달리, 문장을 하나씩 생성해보는 것이 일반적입니다. 'Greedy Search'는 배치 단위로 처리하기 복잡합니다.)
# shuffle=True: 데이터셋에서 '무작위'로 (질문, 답변) 쌍을 하나 뽑아옵니다.
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# [전체 흐름 2: 모델을 '평가 모드'로 전환]
# 'encoder' 모델을 '평가 모드(eval())'로 설정합니다.
# (이유: '훈련 모드(train())'에서만 작동해야 하는 '드롭아웃(Dropout)' 같은 기능들을 비활성화(OFF)시킵니다.
# 이렇게 해야 추론 시 일관성 있고 안정적인 결과가 나옵니다.)
encoder.eval()
# 'decoder' 모델도 '평가 모드(eval())'로 설정합니다.
# (이유: 인코더와 동일하게, 드롭아웃 등을 비활성화하여 일관된 답변을 생성하기 위함입니다.)
decoder.eval()

# 'torch.no_grad()' 블록 안에서 코드를 실행합니다.
# (이유: '평가/추론' 시에는 'loss.backward()' (역전파)를 통한 '기울기 계산'이 전혀 필요 없습니다.
# 이 블록은 기울기 계산을 '방지'하여, 메모리 사용량을 획기적으로 줄이고 계산 속도를 높여줍니다.)
with torch.no_grad():
    # 'dataloader'(배치 크기 1)에서 (질문, 답변) 쌍을 하나 꺼내옵니다.
    # (inp의 모양: [1, 60], tar의 모양: [1, 60])
    # (이 코드는 맨 밑의 'break' 때문에 '단 한 번만' 실행됩니다.)
    for inp, tar in dataloader:
        # [지금 하는 일: (1) 입력(질문)을 텍스트로 변환 (확인용)]
        # (1) inp.long(): [1, 60] 텐서를 정수형으로 변환합니다.
        # (2) .tolist(): 파이토치 텐서를 파이썬 리스트(예: [[15, 10, 4, 0...]])로 변환합니다.
        # (3) sp.decode(...): 'sp'(토크나이저)의 'decode' 기능을 사용해, 이 숫자 리스트를 '사람이 읽을 수 있는 텍스트'로 변환합니다.
        # (결과: input_words는 ['오늘 날씨'] 같은 텍스트가 담긴 리스트가 됩니다.)
        # (이유: 나중에 "입력 텍스트: ..."를 출력하여, 원본 질문과 모델의 답변을 비교하기 위함입니다.)
        input_words = sp.decode(inp.long().tolist())

        # [지금 하는 일: (2) 데이터 준비 (GPU 이동)]
        # 'inp' (질문 텐서, [1, 60])를 '정수(long)' 타입으로 변환하고 'device'(GPU)로 보냅니다.
        # (이유: encoder 모델이 GPU에 있으므로 입력 데이터도 GPU로 보내야 합니다.)
        inp = inp.long().to(device)
        # 'tar' (정답 답변 텐서, [1, 60])도 '정수(long)' 타입으로 변환하고 'device'(GPU)로 보냅니다.
        # (이유: '추론' 시에도 'tar[:, 0]' ([BOS] 토큰)을 디코더의 '첫 입력'으로 사용하기 때문입니다.)
        tar = tar.long().to(device)

        # [지금 하는 일: (3) 인코더 순전파]
        # 준비된 'inp' (질문, [1, 60])를 'encoder' 모델에 통과시킵니다.
        # (결과: 'enc_output' [1, 60, H], 'h' [L, 1, H], 'c' [L, 1, H]가 생성됩니다.)
        # (이것으로 질문 문장에 대한 '요약(h, c)'과 '어텐션용 정보(enc_output)' 준비가 끝났습니다.)
        enc_output, h, c = encoder(inp)

        # [지금 하는 일: (4) 디코더 초기 설정]
        # (주석) 디코더 첫 입력: <bos> 토큰 ([BOS])
        # 'tar' 텐서(정답)의 0번째 토큰, 즉 '[BOS]' 토큰을 'dec_input'(디코더의 첫 입력)으로 설정합니다.
        # (모양: [1]) (배치 크기가 1이므로 0번째 배치의 0번째 토큰)
        # (주석된 코드 torch.tensor([sp.bos_id()]...)도 동일한 역할을 합니다.)
        dec_input = tar[:,0]
        # 디코더의 '첫 번째 은닉 상태'로 '인코더'가 반환한 '최종 은닉 상태(h)'를 설정합니다.
        dec_hidden = h
        # 디코더의 '첫 번째 셀 상태'로 '인코더'가 반환한 '최종 셀 상태(c)'를 설정합니다.
        dec_cell = c
        # (이 코드는 '추론' 코드이므로 'loss'를 실제로 계산/사용하지는 않습니다. 훈련 코드에서 복사된 흔적일 수 있습니다.)
        loss = 0.0

        # [지금 하는 일: (5) 생성된 토큰을 저장할 리스트 초기화]
        # 모델이 '생성한' 단어(토큰 ID)들을 '순서대로' 저장할 빈 리스트를 만듭니다.
        # (이유: 여기에 [5, 8, 3] 처럼 토큰 ID가 쌓이면, 나중에 'sp.decode'로 텍스트로 복원할 수 있습니다.)
        result_tokens = []

        # [전체 흐름 3: 디코더 스텝(단어) 루프 시작 (Autoregressive Inference)]
        # '답변'의 최대 길이(tar.size(1) == max_len == 60)만큼 반복하는 루프를 시작합니다. (t = 1, 2, ... 59)
        # (이유: 답변이 최대 60단어를 넘지 않도록 '안전장치'로 최대 길이를 설정하고, 한 단어씩 생성합니다.)
        for t in range(1, tar.size(1)):
            # [지금 하는 일: (6) 디코더 1 스텝 순전파]
            # 'decoder'의 'forward' 함수를 호출하여 '다음 단어'를 예측합니다.
            # [입력값]
            # 1. 'dec_input': '직전 스텝'에서 예측된 단어 ID (t=1일 때는 [BOS] 토큰, 모양 [1])
            # 2. 'dec_hidden', 'dec_cell': '직전 스텝'의 (은닉, 셀) 상태
            # 3. 'enc_output': '인코더'의 모든 시점 출력 (어텐션 계산용)
            # [반환값]
            # 1. 'pred': '이번 스텝(t)'에서 예측한 '다음 단어'의 점수 분포(logit). (모양: [1, vocab_size])
            # 2. 'dec_hidden', 'dec_cell': '이번 스텝(t)'에서 '갱신된' (은닉, 셀) 상태. (다음 루프(t+1)의 입력으로 들어감)
            pred, dec_hidden, dec_cell = decoder(dec_input, dec_hidden, dec_cell, enc_output)

            # [지금 하는 일: (7) Greedy Search (가장 가능성 높은 단어 선택)]
            # (주석) 예측된 단어 ID (차원: [batch_size, vocab_size] -> argmax dim=1)
            # (1) pred[0]: 'pred' (모양 [1, vocab_size])에서 배치 차원을 제거(또는 선택)합니다. (모양: [vocab_size])
            # (2) .argmax(dim=-1): 'vocab_size'개의 점수(logit) 중에서 '가장 높은 점수'를 가진 단어의 '인덱스(ID)'를 찾습니다.
            # (이것이 'Greedy Search' (탐욕법)입니다. "지금 당장 가장 좋아보이는 단어 1개만 선택한다"는 의미입니다.)
            # (3) .item(): 텐서(예: tensor(5))에서 파이썬 숫자(예: 5)만 추출합니다.
            # (결과: 'predicted_id'에는 모델이 '이번 스텝(t)'에 생성하기로 '결정한' 단어의 ID(숫자)가 저장됩니다.)
            predicted_id = pred[0].argmax(dim=-1).item()

            # [지금 하는 일: (8) 생성된 토큰 저장]
            # (주석) result에 누적
            # 모델이 방금 선택한 단어 ID('predicted_id')를 'result_tokens' 리스트의 '맨 뒤'에 추가합니다.
            result_tokens.append(predicted_id)

            # [지금 하는 일: (9) 종료 조건 확인]
            # (주석) <end> 토큰이면 중단
            # '만약' 모델이 방금 생성한 단어('predicted_id')가 'sp.eos_id()' ([EOS], 문장 끝) 토큰 ID와 '같다면',
            if predicted_id == sp.eos_id():
                # "답변 생성이 완료되었다"고 판단하여, 'for t ...' (디코더 스텝 루프)를 '즉시 중단(break)'합니다.
                # (이유: [EOS] 토큰 뒤에는 더 이상 유의미한 단어가 생성되지 않으므로, 여기서 생성을 멈춰야 합니다.)
                break

            # [지금 하는 일: (10) 다음 스텝 입력 준비 (*** 훈련과 가장 큰 차이 ***)]
            # (주석) 다음 시점 디코더 입력으로 predicted_id
            # '만약' [EOS]가 아니라서 루프가 계속된다면, '다음 스텝(t+1)'의 '입력'('dec_input')으로,
            # '훈련' 때처럼 정답('tar[:, t]')을 주는 것이 아니라,
            # '방금 모델이 예측한' 'predicted_id'를 '새로운 텐서'로 만들어서 설정합니다.
            # (이것이 '추론(Inference)'과 '훈련(Teacher Forcing)'의 '가장 큰 차이점'입니다. 모델이 자신의 예측을 되먹임(autoregressive)합니다.)
            # (dtype=torch.long, device=device: 텐서를 정수형으로, 그리고 GPU(device) 위에 생성합니다. 모양: [1])'
            # (의미)
            # "학생(모델)아, 네가 (t)스텝에서 'I'라고 '스스로 예측'했지? (predicted_id)"
            # "좋아, 이제 그 '네가 뱉은 말(I)'을 t+1 스텝의 '다음 힌트'로 '스스로' 사용해봐."
            # "그리고 그 '네가 뱉은 말(I)'에 이어서 나올 '다음 단어'를 예측해 봐."
            #
            # (목적)
            # '실제 챗봇'이 작동하는 방식입니다.
            # 챗봇은 사용자가 무슨 말을 할지 '정답지(tar)'를 미리 알 수 없습니다.
            # 오직 '자신(모델)'이 '방금 생성한 단어'를 '다음 입력'으로 '되먹임(Autoregressive)'하면서
            # 문장이 끝날 때([EOS])까지 스스로 '이어나가야' 합니다.
            dec_input = torch.tensor([predicted_id], dtype=torch.long, device=device)

        # [전체 흐름 4: 최종 결과 변환 및 출력]
        # (디코더 스텝 루프(for t...)가 (break로) 종료된 후)
        # (주석) 예측된 토큰들을 문자열로 변환
        # 'result_tokens' (모델이 생성한 ID 리스트, 예: [5, 8, 3])를 'sp.decode'에 넣어,
        # '사람이 읽을 수 있는 텍스트'('result_words', 예: "네 안녕하세요[EOS]")로 변환합니다.
        result_words = sp.decode(result_tokens)

        # [지금 하는 일: (11) 결과 출력]
        # 맨 처음에 'sp.decode'로 변환해 두었던 '원본 질문 텍스트'('input_words[0]')를 출력합니다.
        print(f'입력 텍스트: {input_words[0]}')
        # 'sp.decode'로 변환한 '모델이 생성한 답변 텍스트'('result_words')를 출력합니다.
        print(f'출력 텍스트: {result_words}')

        # 'for inp, tar in dataloader:' (데이터로더 루프)를 '즉시 중단(break)'합니다.
        # (이유: 이 코드의 목적은 '하나'의 샘플에 대해 질문/답변을 생성해 '테스트'하는 것이므로, 1개만 실행하고 종료합니다.)
        break

입력 텍스트: 해도 너무 하지. 오늘 주어진 업무는 도저히 혼자 할 수 있는 양이 아니잖아.
출력 텍스트: 어떤 점이 가장 힘드신가요?
